# DAY 2 - 배터리 수명 예측 모델링

## 1. 데이터 분할 고정

DAY 2의 첫 단계는 모델을 보기 전에 데이터의 역할을 고정하는 것이다.

- **Train**: Batch 1의 정책 단위 Group Cross-Validation에 사용
- **Valid**: Batch 1의 고정 policy Hold-out
- **Test**: Batch 2의 수명 라벨 보유 셀, 최종 모델 확정 후 1회 평가
- **Additional**: Batch 3, 선택적인 외부 일반화 점검

같은 충전 policy의 셀이 Train과 Valid에 동시에 들어가면 동일 실험조건을 공유한 셀이 양쪽에 섞여 성능이 낙관적으로 보일 수 있다. 따라서 Batch 1은 policy를 group으로 사용하여 분리한다. 최초로 생성한 split manifest를 저장하고 이후 실행에서는 같은 분할을 다시 불러온다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42
VALID_SIZE = 0.20


def find_project_root(start: Path) -> Path:
    # 현재 위치가 프로젝트 루트 또는 notebooks 폴더여도 루트를 찾는다.
    start = start.resolve()
    for candidate in (start, *start.parents):
        feature_path = candidate / "results" / "battery_eda" / "cell_features.csv"
        if feature_path.exists():
            return candidate
    raise FileNotFoundError(
        "results/battery_eda/cell_features.csv를 포함한 프로젝트 루트를 찾지 못했습니다."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
FEATURE_PATH = PROJECT_ROOT / "results" / "battery_eda" / "cell_features.csv"
MODEL_RESULT_DIR = PROJECT_ROOT / "results" / "modeling"
SPLIT_MANIFEST_PATH = MODEL_RESULT_DIR / "split_manifest_v1.csv"

MODEL_RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"FEATURE_PATH: {FEATURE_PATH}")
print(f"SPLIT_MANIFEST_PATH: {SPLIT_MANIFEST_PATH}")

PROJECT_ROOT: /Users/jungseungwoo/Desktop/data-mini-project
FEATURE_PATH: /Users/jungseungwoo/Desktop/data-mini-project/results/battery_eda/cell_features.csv
SPLIT_MANIFEST_PATH: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/split_manifest_v1.csv


### 1.1 모델링 데이터 로드 및 기본 검증

cycle_life가 없는 셀은 목표변수 기반 회귀 학습과 성능 평가에서 제외한다. 목표변수 Y는 평균이나 중앙값으로 대체하지 않는다. Batch 2는 이 단계에서 분할 목록만 고정하고, 예측과 성능 계산은 모델·feature·하이퍼파라미터를 확정한 뒤 수행한다.

In [2]:
df = pd.read_csv(FEATURE_PATH)

required_columns = {"batch", "cell_id", "policy", "cycle_life"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {sorted(missing_columns)}")

if df["cell_id"].duplicated().any():
    duplicated = df.loc[df["cell_id"].duplicated(False), "cell_id"].tolist()
    raise ValueError(f"cell_id가 중복되었습니다: {duplicated}")

unlabeled_df = df[df["cycle_life"].isna()].copy()
labeled_df = df[df["cycle_life"].notna()].copy()

batch1_df = labeled_df[labeled_df["batch"].eq("Batch 1")].copy()
batch2_df = labeled_df[labeled_df["batch"].eq("Batch 2")].copy()
batch3_df = labeled_df[labeled_df["batch"].eq("Batch 3")].copy()

assert len(batch1_df) == 46, f"Batch 1 라벨 셀 수 확인 필요: {len(batch1_df)}"
assert len(batch2_df) == 39, f"Batch 2 라벨 셀 수 확인 필요: {len(batch2_df)}"
assert len(batch3_df) == 44, f"Batch 3 라벨 셀 수 확인 필요: {len(batch3_df)}"

data_overview = pd.DataFrame({
    "raw_cells": df.groupby("batch")["cell_id"].size(),
    "labeled_cells": labeled_df.groupby("batch")["cell_id"].size(),
    "unlabeled_cells": unlabeled_df.groupby("batch")["cell_id"].size(),
}).fillna(0).astype(int)

display(data_overview)

,raw_cells,labeled_cells,unlabeled_cells
batch,,,
Batch 1,46,46,0
Batch 2,47,39,8
Batch 3,46,44,2


### 1.2 Batch 1 policy 단위 Hold-out 고정

최초 실행에서는 GroupShuffleSplit(random_state=42)로 분할하고 셀 ID, policy, 역할을 split_manifest_v1.csv에 저장한다. 이후 실행에서는 이 파일을 읽으므로 모델 결과를 보고 유리한 Hold-out으로 바꾸는 것을 방지한다.

In [3]:
if SPLIT_MANIFEST_PATH.exists():
    split_manifest = pd.read_csv(SPLIT_MANIFEST_PATH)
    print("기존 split manifest를 재사용합니다.")
else:
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=VALID_SIZE,
        random_state=RANDOM_STATE,
    )

    train_idx, valid_idx = next(
        splitter.split(
            X=batch1_df,
            y=batch1_df["cycle_life"],
            groups=batch1_df["policy"],
        )
    )

    batch1_split = batch1_df[["batch", "cell_id", "policy"]].copy()
    batch1_split["split"] = ""
    batch1_split.iloc[train_idx, batch1_split.columns.get_loc("split")] = "train"
    batch1_split.iloc[valid_idx, batch1_split.columns.get_loc("split")] = "valid"

    batch2_split = batch2_df[["batch", "cell_id", "policy"]].copy()
    batch2_split["split"] = "test"

    batch3_split = batch3_df[["batch", "cell_id", "policy"]].copy()
    batch3_split["split"] = "additional"

    split_manifest = pd.concat(
        [batch1_split, batch2_split, batch3_split],
        ignore_index=True,
    )
    split_manifest["random_state"] = RANDOM_STATE
    split_manifest.to_csv(SPLIT_MANIFEST_PATH, index=False)
    print("새 split manifest를 생성하고 저장했습니다.")

display(split_manifest.head())

기존 split manifest를 재사용합니다.


,batch,cell_id,policy,split,random_state
0,Batch 1,b1c0,3_6C-80PER_3_6C,valid,42
1,Batch 1,b1c1,3_6C-80PER_3_6C,valid,42
2,Batch 1,b1c2,3_6C-80PER_3_6C,valid,42
3,Batch 1,b1c3,4C-80PER_4C,train,42
4,Batch 1,b1c4,4C-80PER_4C,train,42


### 1.3 분할 검증

저장된 manifest와 현재 데이터가 일치하는지, Train과 Valid에 동일 policy가 섞이지 않았는지 확인한다. 이 검증을 통과하지 않으면 이후 모델링을 진행하지 않는다.

In [4]:
# 저장된 분할 파일 자체의 무결성을 먼저 검사한다.
ALLOWED_SPLITS = {"train", "valid", "test", "additional"}

assert "split" in split_manifest.columns, "manifest에 split 컬럼이 없습니다."
assert "random_state" in split_manifest.columns, (
    "manifest에 random_state 컬럼이 없습니다."
)
assert split_manifest["split"].notna().all(), "split 역할이 비어 있는 셀이 있습니다."
assert split_manifest["cell_id"].is_unique, "manifest에 중복 cell_id가 있습니다."

unexpected_splits = set(split_manifest["split"]) - ALLOWED_SPLITS
assert not unexpected_splits, (
    f"허용되지 않은 split 이름이 있습니다: {sorted(unexpected_splits)}"
)
assert split_manifest["random_state"].eq(RANDOM_STATE).all(), (
    "manifest의 random_state가 현재 설정과 다릅니다."
)

# Batch별로 허용된 역할도 검사한다.
EXPECTED_SPLITS_BY_BATCH = {
    "Batch 1": {"train", "valid"},
    "Batch 2": {"test"},
    "Batch 3": {"additional"},
}

for batch_name, expected_splits in EXPECTED_SPLITS_BY_BATCH.items():
    actual_splits = set(
        split_manifest.loc[
            split_manifest["batch"].eq(batch_name),
            "split",
        ]
    )
    assert actual_splits == expected_splits, (
        f"{batch_name} 역할 오류: expected={expected_splits}, "
        f"actual={actual_splits}"
    )

print("Split manifest 안전검사 통과")

expected_cells = set(labeled_df["cell_id"])
manifest_cells = set(split_manifest["cell_id"])

if expected_cells != manifest_cells:
    missing_in_manifest = sorted(expected_cells - manifest_cells)
    extra_in_manifest = sorted(manifest_cells - expected_cells)
    raise ValueError(
        "현재 데이터와 split manifest가 다릅니다. "
        f"누락={missing_in_manifest}, 추가={extra_in_manifest}"
    )

train_ids = split_manifest.loc[split_manifest["split"].eq("train"), "cell_id"]
valid_ids = split_manifest.loc[split_manifest["split"].eq("valid"), "cell_id"]
test_ids = split_manifest.loc[split_manifest["split"].eq("test"), "cell_id"]

train_df = batch1_df[batch1_df["cell_id"].isin(train_ids)].copy()
valid_df = batch1_df[batch1_df["cell_id"].isin(valid_ids)].copy()
test_df = batch2_df[batch2_df["cell_id"].isin(test_ids)].copy()

train_policies = set(train_df["policy"])
valid_policies = set(valid_df["policy"])
policy_overlap = train_policies & valid_policies

assert not policy_overlap, f"Train/Valid policy 누수: {sorted(policy_overlap)}"
assert set(train_df["cell_id"]).isdisjoint(valid_df["cell_id"]), "셀 ID 누수 발생"
assert len(train_df) + len(valid_df) == len(batch1_df), "Batch 1 분할 개수 불일치"
assert len(test_df) == len(batch2_df), "Batch 2 Test 개수 불일치"

split_summary = pd.DataFrame({
    "split": ["Train (Batch 1)", "Valid (Batch 1 Hold-out)", "Test (Batch 2)"],
    "cells": [len(train_df), len(valid_df), len(test_df)],
    "policies": [
        train_df["policy"].nunique(),
        valid_df["policy"].nunique(),
        test_df["policy"].nunique(),
    ],
    "target_min": [
        train_df["cycle_life"].min(),
        valid_df["cycle_life"].min(),
        np.nan,
    ],
    "target_median": [
        train_df["cycle_life"].median(),
        valid_df["cycle_life"].median(),
        np.nan,
    ],
    "target_max": [
        train_df["cycle_life"].max(),
        valid_df["cycle_life"].max(),
        np.nan,
    ],
})

print(f"Train/Valid policy overlap: {policy_overlap}")
print(f"Split manifest 고정 경로: {SPLIT_MANIFEST_PATH}")
display(split_summary)

Split manifest 안전검사 통과
Train/Valid policy overlap: set()
Split manifest 고정 경로: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/split_manifest_v1.csv


,split,cells,policies,target_min,target_median,target_max
0,Train (Batch 1),35,18,534.0,860.0,1227.0
1,Valid (Batch 1 Hold-out),11,5,691.0,857.0,1190.0
2,Test (Batch 2),39,12,NaN,NaN,NaN


### 1.4 이후 모델링에서 지킬 원칙

1. 후보 feature·모델·하이퍼파라미터는 Train 내부 Group CV 결과로 비교한다.
2. Valid는 최종 후보 확인용으로 사용하며 반복적으로 확인하면서 전략을 바꾸지 않는다.
3. Batch 2 Test는 모델을 완전히 고정한 후 한 번만 평가한다.
4. Test 결과를 확인한 뒤 feature나 모델을 다시 선택하지 않는다.
5. split manifest를 삭제하거나 재생성하지 않고 Github에 함께 기록한다.

In [5]:
# 다음 섹션에서 사용할 객체만 준비한다. 아직 Batch 2 성능은 계산하지 않는다.
TARGET = "cycle_life"
GROUP_COLUMN = "policy"

y_train = train_df[TARGET].copy()
y_valid = valid_df[TARGET].copy()
groups_train = train_df[GROUP_COLUMN].copy()

print("데이터 역할 고정 완료")
print(f"Train cells: {len(train_df)}")
print(f"Valid cells: {len(valid_df)}")
print(f"Locked Test cells: {len(test_df)} - 최종 모델 확정 전 평가 금지")

데이터 역할 고정 완료
Train cells: 35
Valid cells: 11
Locked Test cells: 39 - 최종 모델 확정 전 평가 금지


## 2. 데이터 누수 변수 제거

모델 입력에는 예측 시점인 초기 100사이클 안에서 알 수 있는 수치형 정보만 사용한다. 목표값, 목표값에서 파생된 레이블, 전체 수명 또는 EOL을 알아야 계산할 수 있는 변수, 식별자 및 원본 policy 문자열은 제거한다.

핵심 원칙:

- cycle_life는 목표변수 Y이며 X에 포함하지 않는다.
- life_group과 label_550은 cycle_life에서 파생되므로 제거한다.
- summary_cycles와 n_cycles는 전체 관측 길이로 수명을 직접 암시하므로 제거한다.
- knee 변수는 수명 후반부의 전체 곡선이 필요하므로 제거한다.
- cell_id와 batch는 식별·평가 그룹일 뿐 예측변수가 아니다.
- policy는 분할용 group으로만 사용하고 모델 입력에는 넣지 않는다.
- Y 결측값은 대체하지 않고 지도학습 대상에서 제외한다.

In [6]:
# Y가 없는 셀은 회귀 학습과 성능 평가에서 제외한다.
# 목표변수 Y에는 어떠한 대체도 수행하지 않는다.
model_df = df.dropna(subset=["cycle_life"]).copy()

TARGET_COLUMN = "cycle_life"
SPLIT_GROUP_COLUMN = "policy"

IDENTIFIER_COLUMNS = {
    "cell_id",
    "batch",
}

TARGET_DERIVED_COLUMNS = {
    "cycle_life",
    "life_group",
    "label_550",
}

FUTURE_INFORMATION_COLUMNS = {
    "n_cycles",
    "summary_cycles",
    "knee_cycle_proxy",
    "knee_fraction_proxy",
}

RAW_POLICY_COLUMNS = {
    "policy",
    "policy_readable",
    "protocol_type",
}

EXCLUDED_COLUMNS = (
    IDENTIFIER_COLUMNS
    | TARGET_DERIVED_COLUMNS
    | FUTURE_INFORMATION_COLUMNS
    | RAW_POLICY_COLUMNS
)

leakage_reasons = {
    "cell_id": "셀 식별자",
    "batch": "평가 그룹; 배치 정보를 수명 신호로 학습할 위험",
    "cycle_life": "목표변수 Y",
    "life_group": "cycle_life에서 파생된 그룹",
    "label_550": "cycle_life에서 파생된 분류 레이블",
    "n_cycles": "전체 관측 길이; 수명의 직접 대리변수",
    "summary_cycles": "전체 관측 길이; 수명의 직접 대리변수",
    "knee_cycle_proxy": "전체 열화곡선이 필요한 미래정보",
    "knee_fraction_proxy": "전체 수명으로 정규화된 미래정보",
    "policy": "분할용 group 전용",
    "policy_readable": "원본 policy 문자열의 표현",
    "protocol_type": "policy에서 파생된 범주형 그룹",
}

leakage_audit = pd.DataFrame([
    {
        "column": column,
        "exists_in_data": column in model_df.columns,
        "reason": leakage_reasons[column],
    }
    for column in sorted(leakage_reasons)
])

print(f"전체 셀 수: {len(df)}")
print(f"Y 라벨 보유 셀 수: {len(model_df)}")
print(f"Y 결측으로 지도학습에서 제외된 셀 수: {df[TARGET_COLUMN].isna().sum()}")
display(leakage_audit)

전체 셀 수: 139
Y 라벨 보유 셀 수: 129
Y 결측으로 지도학습에서 제외된 셀 수: 10


,column,exists_in_data,reason
0,batch,True,평가 그룹; 배치 정보를 수명 신호로 학습할 위험
1,cell_id,True,셀 식별자
2,cycle_life,True,목표변수 Y
3,knee_cycle_proxy,True,전체 열화곡선이 필요한 미래정보
4,knee_fraction_proxy,True,전체 수명으로 정규화된 미래정보
5,label_550,True,cycle_life에서 파생된 분류 레이블
6,life_group,True,cycle_life에서 파생된 그룹
7,n_cycles,False,전체 관측 길이; 수명의 직접 대리변수
8,policy,True,분할용 group 전용
9,policy_readable,True,원본 policy 문자열의 표현


### 2.1 초기 100사이클 후보변수 allowlist

누수 컬럼을 이름으로만 제거하면 새로운 미래정보 컬럼이 추가됐을 때 실수로 모델에 들어갈 수 있다. 따라서 제거 목록과 함께 초기 100사이클에 사용 가능한 후보변수 목록을 명시적으로 관리한다.

아래 목록은 누수 여부만 판단한 후보군이다. 실제 Feature Set A·B·C와 최종 변수 선택은 다음 단계에서 Batch 1 Train의 Group CV로 결정한다.

In [7]:
EARLY_FEATURE_CANDIDATES = [
    # 충전조건: 실험 시작 전에 알 수 있는 수치형 조건
    "c_rate_1",
    "switch_soc",
    "c_rate_2",

    # 초기 방전용량
    "Qd_mean_2_10",
    "Qd_10",
    "Qd_100",
    "Qd_change_10_100",
    "Qd_slope_10_100",

    # 초기 내부저항
    "IR_mean_2_100",
    "IR_100",
    "IR_slope_10_100",

    # 초기 온도
    "Tavg_mean_2_100",
    "Tmax_max_2_100",

    # 초기 충전시간
    "charge_time_mean_2_100",
    "charge_time_std_2_100",

    # Cycle 10과 100의 Delta Q(V)
    "deltaQ_min",
    "deltaQ_mean",
    "deltaQ_abs_mean",
    "deltaQ_range",
    "deltaQ_var",
    "deltaQ_logvar",
    "deltaQ_skew",
    "deltaQ_kurtosis",
]

missing_candidate_columns = sorted(
    set(EARLY_FEATURE_CANDIDATES) - set(model_df.columns)
)
if missing_candidate_columns:
    raise ValueError(
        f"초기 후보변수 컬럼이 데이터에 없습니다: {missing_candidate_columns}"
    )

leakage_in_candidates = sorted(
    set(EARLY_FEATURE_CANDIDATES) & EXCLUDED_COLUMNS
)
assert not leakage_in_candidates, (
    f"후보변수에 누수 컬럼이 포함되었습니다: {leakage_in_candidates}"
)

non_numeric_candidates = model_df[EARLY_FEATURE_CANDIDATES].select_dtypes(
    exclude=[np.number]
).columns.tolist()
assert not non_numeric_candidates, (
    f"후보변수에 비수치형 컬럼이 포함되었습니다: {non_numeric_candidates}"
)

print(f"초기 100사이클 후보변수 수: {len(EARLY_FEATURE_CANDIDATES)}")
print("누수 컬럼과 후보변수의 교집합: 없음")
display(pd.DataFrame({"candidate_feature": EARLY_FEATURE_CANDIDATES}))

초기 100사이클 후보변수 수: 23
누수 컬럼과 후보변수의 교집합: 없음


,candidate_feature
0,c_rate_1
1,switch_soc
2,c_rate_2
3,Qd_mean_2_10
4,Qd_10
5,Qd_100
6,Qd_change_10_100
7,Qd_slope_10_100
8,IR_mean_2_100
9,IR_100


### 2.2 고정 분할에 누수 제거 적용

Section 1에서 고정한 Train·Valid·Test에 동일한 후보변수 allowlist를 적용한다. Test의 설명변수 X는 준비할 수 있지만, 목표값 y_test는 최종 모델이 확정될 때까지 별도 객체로 만들지 않는다. 수치형 X의 결측치 대체는 이후 Pipeline 내부의 Train fold에서만 수행한다.

In [8]:
X_train_all = train_df[EARLY_FEATURE_CANDIDATES].copy()
X_valid_all = valid_df[EARLY_FEATURE_CANDIDATES].copy()
X_test_locked = test_df[EARLY_FEATURE_CANDIDATES].copy()

y_train = train_df[TARGET_COLUMN].copy()
y_valid = valid_df[TARGET_COLUMN].copy()
groups_train = train_df[SPLIT_GROUP_COLUMN].copy()

# 최종 모델 확정 전에는 아래 코드를 실행하거나 y_test를 생성하지 않는다.
# y_test = test_df[TARGET_COLUMN].copy()

assert TARGET_COLUMN not in X_train_all.columns
assert TARGET_COLUMN not in X_valid_all.columns
assert TARGET_COLUMN not in X_test_locked.columns
assert not (set(X_train_all.columns) & EXCLUDED_COLUMNS)
assert y_train.notna().all() and y_valid.notna().all()

missing_summary = pd.DataFrame({
    "Train_missing": X_train_all.isna().sum(),
    "Valid_missing": X_valid_all.isna().sum(),
    "Test_missing": X_test_locked.isna().sum(),
})

split_feature_summary = pd.DataFrame({
    "split": ["Train", "Valid", "Test locked"],
    "rows": [len(X_train_all), len(X_valid_all), len(X_test_locked)],
    "features": [
        X_train_all.shape[1],
        X_valid_all.shape[1],
        X_test_locked.shape[1],
    ],
    "target_created": [True, True, False],
})

display(split_feature_summary)
display(missing_summary.loc[missing_summary.sum(axis=1).gt(0)])

print("누수 제거 검증 완료")
print("Y 결측치 대체 없음")
print("X 결측치는 아직 대체하지 않음 - 이후 Pipeline의 Train fold에서 처리")
print("Batch 2 y_test는 아직 생성하지 않음")

,split,rows,features,target_created
0,Train,35,23,True
1,Valid,11,23,True
2,Test locked,39,23,False


,Train_missing,Valid_missing,Test_missing
IR_slope_10_100,0,0,6


누수 제거 검증 완료
Y 결측치 대체 없음
X 결측치는 아직 대체하지 않음 - 이후 Pipeline의 Train fold에서 처리
Batch 2 y_test는 아직 생성하지 않음


### 2.3 누수 제거 결과

- 모델링 라벨을 가진 129개 셀만 지도학습 대상으로 유지했다.
- 목표값 파생 변수, 전체 수명 대리변수, knee 변수, 식별자 및 raw policy를 X에서 차단했다.
- 모델 입력 후보는 초기 100사이클에서 계산 가능한 수치형 변수로 제한했다.
- policy는 Train 내부 Group CV와 Hold-out 분할에만 사용한다.
- Batch 2 목표값은 최종 모델 확정 전까지 평가 코드에서 사용하지 않는다.
- 다음 단계에서는 이 후보군으로 Feature Set을 정의하고 Batch 1 Group CV로 비교한다.

### 2.4 Policy Group Cross-Validation 고정

고정 Hold-out과 별개로 후보 feature와 모델은 Batch 1 Train 내부에서 5-fold GroupKFold로 비교한다. 일반 KFold는 같은 policy의 셀을 서로 다른 fold에 섞을 수 있으므로 사용하지 않는다.

GroupKFold는 같은 policy에 속한 모든 셀을 하나의 fold로 함께 이동시킨다. 아래 코드는 실제 모델 평가 전에 모든 fold에서 학습 policy와 검증 policy의 교집합이 비어 있는지 확인한다.

다음 모델 비교에서는 group_cv와 groups_train을 그대로 사용한다. GroupKFold 자체는 random_state를 사용하지 않으며, 현재 데이터와 group 순서에서 결정적으로 동작한다.

In [9]:
from sklearn.model_selection import GroupKFold

N_GROUP_FOLDS = 5

assert groups_train.nunique() >= N_GROUP_FOLDS, (
    f"policy 그룹 수 {groups_train.nunique()}개로 "
    f"{N_GROUP_FOLDS}-fold를 만들 수 없습니다."
)

group_cv = GroupKFold(n_splits=N_GROUP_FOLDS)
cv_splits = list(
    group_cv.split(
        X=X_train_all,
        y=y_train,
        groups=groups_train,
    )
)

fold_audit_rows = []

for fold_number, (fold_train_idx, fold_valid_idx) in enumerate(
    cv_splits,
    start=1,
):
    fold_train_groups = set(groups_train.iloc[fold_train_idx])
    fold_valid_groups = set(groups_train.iloc[fold_valid_idx])
    fold_group_overlap = fold_train_groups & fold_valid_groups

    assert not fold_group_overlap, (
        f"Fold {fold_number} policy 누수: {sorted(fold_group_overlap)}"
    )
    assert set(fold_train_idx).isdisjoint(fold_valid_idx), (
        f"Fold {fold_number} 행 인덱스 누수"
    )

    fold_audit_rows.append({
        "fold": fold_number,
        "train_cells": len(fold_train_idx),
        "valid_cells": len(fold_valid_idx),
        "train_policies": len(fold_train_groups),
        "valid_policies": len(fold_valid_groups),
        "policy_overlap": len(fold_group_overlap),
        "valid_target_min": y_train.iloc[fold_valid_idx].min(),
        "valid_target_median": y_train.iloc[fold_valid_idx].median(),
        "valid_target_max": y_train.iloc[fold_valid_idx].max(),
    })

group_cv_audit = pd.DataFrame(fold_audit_rows)

display(group_cv_audit)
print(f"GroupKFold 준비 완료: {N_GROUP_FOLDS} folds")
print(f"Train policy 수: {groups_train.nunique()}")
print("모든 fold의 Train/Validation policy overlap: 0")
print("다음 cross_validate 호출에서 groups=groups_train을 반드시 전달할 것")

,fold,train_cells,valid_cells,train_policies,valid_policies,policy_overlap,valid_target_min,valid_target_median,valid_target_max
0,1,27,8,14,4,0,534.0,626.0,870.0
1,2,27,8,14,4,0,651.0,870.0,1227.0
2,3,28,7,14,4,0,842.0,902.0,1074.0
3,4,29,6,15,3,0,617.0,821.0,1017.0
4,5,29,6,15,3,0,648.0,885.0,1054.0


GroupKFold 준비 완료: 5 folds
Train policy 수: 18
모든 fold의 Train/Validation policy overlap: 0
다음 cross_validate 호출에서 groups=groups_train을 반드시 전달할 것


#### 다음 모델 섹션에서 사용할 형태

모델 Pipeline과 scoring이 정의되면 다음 형태로 평가한다.

    cv_result = cross_validate(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        groups=groups_train,
        cv=group_cv,
        scoring=scoring,
    )

가장 중요한 인자는 groups=groups_train이다. 이를 생략하면 GroupKFold가 각 셀의 policy를 전달받지 못한다.

## 3. Feature Set 사전 정의

Batch 2 Test 결과를 확인한 뒤 유리한 변수 조합으로 바꾸는 것을 방지하기 위해 비교할 Feature Set을 모델 학습 전에 고정한다.

- **Feature Set A**: DAY1에서 가장 강하고 일관된 신호였던 deltaQ_logvar 단독
- **Feature Set B**: ΔQ(V)의 분포 형태를 나타내는 skew·kurtosis 추가
- **Feature Set C**: 온도·충전시간·첫 단계 C-rate를 추가한 탐색용 비교군

Feature Set A를 기준으로 두고, B와 C는 Batch 1 Train의 policy Group CV에서 복잡도 증가가 실제 성능과 안정성을 개선하는지 확인하는 비교군으로만 사용한다. Feature Set 정의와 상관검사는 Batch 1 Train만 사용하며 Batch 2 통계는 확인하지 않는다.

In [10]:
import json

FEATURE_SET_VERSION = "v1"
FEATURE_SET_MANIFEST_PATH = (
    MODEL_RESULT_DIR / f"feature_sets_{FEATURE_SET_VERSION}.json"
)

features_a = [
    "deltaQ_logvar",
]

features_b = [
    "deltaQ_logvar",
    "deltaQ_skew",
    "deltaQ_kurtosis",
]

features_c = [
    "deltaQ_logvar",
    "Tavg_mean_2_100",
    "charge_time_mean_2_100",
    "c_rate_1",
]

FEATURE_SETS = {
    "A_deltaQ_logvar": features_a,
    "B_deltaQ_shape": features_b,
    "C_exploratory_aux": features_c,
}

# JSON에서 안정적으로 비교할 수 있도록 정렬 가능한 기본 타입만 저장한다.
feature_set_config = {
    "version": FEATURE_SET_VERSION,
    "selection_data": "Batch 1 Train only",
    "test_data_used_for_selection": False,
    "sets": FEATURE_SETS,
}

if FEATURE_SET_MANIFEST_PATH.exists():
    with FEATURE_SET_MANIFEST_PATH.open("r", encoding="utf-8") as file:
        saved_feature_set_config = json.load(file)
    assert saved_feature_set_config == feature_set_config, (
        "저장된 Feature Set과 현재 코드가 다릅니다. "
        "Test 결과 확인 후 변수 조합을 바꾸지 않았는지 확인하세요."
    )
    print("기존 Feature Set manifest를 재사용합니다.")
else:
    with FEATURE_SET_MANIFEST_PATH.open("w", encoding="utf-8") as file:
        json.dump(
            feature_set_config,
            file,
            ensure_ascii=False,
            indent=2,
        )
    print("Feature Set manifest를 최초 생성했습니다.")

for set_name, columns in FEATURE_SETS.items():
    assert len(columns) == len(set(columns)), (
        f"{set_name}에 중복 변수가 있습니다."
    )
    assert set(columns).issubset(EARLY_FEATURE_CANDIDATES), (
        f"{set_name}에 초기 후보군 밖의 변수가 있습니다: "
        f"{sorted(set(columns) - set(EARLY_FEATURE_CANDIDATES))}"
    )
    assert not (set(columns) & EXCLUDED_COLUMNS), (
        f"{set_name}에 누수 변수가 포함되었습니다."
    )
    assert "deltaQ_logvar" in columns, (
        f"{set_name}에 기준변수 deltaQ_logvar가 없습니다."
    )

feature_set_table = pd.DataFrame([
    {
        "feature_set": set_name,
        "n_features": len(columns),
        "features": ", ".join(columns),
        "role": {
            "A_deltaQ_logvar": "기준 모델",
            "B_deltaQ_shape": "Delta Q 형태정보 추가",
            "C_exploratory_aux": "보조변수 탐색 비교군",
        }[set_name],
    }
    for set_name, columns in FEATURE_SETS.items()
])

display(feature_set_table)
print(f"Feature Set 고정 경로: {FEATURE_SET_MANIFEST_PATH}")

기존 Feature Set manifest를 재사용합니다.


,feature_set,n_features,features,role
0,A_deltaQ_logvar,1,deltaQ_logvar,기준 모델
1,B_deltaQ_shape,3,"deltaQ_logvar, deltaQ_skew, deltaQ_kurtosis",Delta Q 형태정보 추가
2,C_exploratory_aux,4,"deltaQ_logvar, Tavg_mean_2_100, charge_time_me...",보조변수 탐색 비교군


Feature Set 고정 경로: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/feature_sets_v1.json


### 3.1 Feature Set B 중복성 확인

deltaQ_skew와 deltaQ_kurtosis가 기준변수 deltaQ_logvar와 지나치게 겹치면 변수를 추가할 실익이 작고 선형모델의 계수도 불안정해질 수 있다.

Batch 1 Train에서 Pearson과 Spearman 상관을 모두 확인한다. 절대 상관계수 0.85 이상을 심한 중복의 검토 기준으로 표시하지만, 이 단계에서 Test나 Valid 성능을 보고 변수를 제거하지 않는다. 최종 채택 여부는 Train Group CV 결과로 결정한다.

In [11]:
HIGH_CORRELATION_THRESHOLD = 0.85

set_b_train = X_train_all[features_b].copy()
set_b_pearson = set_b_train.corr(method="pearson")
set_b_spearman = set_b_train.corr(method="spearman")


def find_high_correlation_pairs(correlation_matrix, threshold):
    pairs = []
    columns = correlation_matrix.columns.tolist()
    for left_index, left_column in enumerate(columns):
        for right_column in columns[left_index + 1:]:
            correlation = correlation_matrix.loc[left_column, right_column]
            if abs(correlation) >= threshold:
                pairs.append({
                    "feature_1": left_column,
                    "feature_2": right_column,
                    "correlation": correlation,
                    "abs_correlation": abs(correlation),
                })
    return pd.DataFrame(pairs)


set_b_high_pearson = find_high_correlation_pairs(
    set_b_pearson,
    HIGH_CORRELATION_THRESHOLD,
)
set_b_high_spearman = find_high_correlation_pairs(
    set_b_spearman,
    HIGH_CORRELATION_THRESHOLD,
)

print("Feature Set B - Pearson correlation (Batch 1 Train only)")
display(set_b_pearson.round(3))

print("Feature Set B - Spearman correlation (Batch 1 Train only)")
display(set_b_spearman.round(3))

if set_b_high_pearson.empty and set_b_high_spearman.empty:
    print(
        "절대 상관계수 0.85 이상의 심한 중복은 없습니다. "
        "Feature Set B를 CV 비교 후보로 유지합니다."
    )
else:
    print("심한 중복 후보가 발견되었습니다. Group CV 결과와 함께 판단합니다.")
    display(set_b_high_pearson)
    display(set_b_high_spearman)

Feature Set B - Pearson correlation (Batch 1 Train only)


,deltaQ_logvar,deltaQ_skew,deltaQ_kurtosis
deltaQ_logvar,1.000,0.691,-0.078
deltaQ_skew,0.691,1.000,-0.085
deltaQ_kurtosis,-0.078,-0.085,1.000


Feature Set B - Spearman correlation (Batch 1 Train only)


,deltaQ_logvar,deltaQ_skew,deltaQ_kurtosis
deltaQ_logvar,1.000,0.668,-0.264
deltaQ_skew,0.668,1.000,-0.246
deltaQ_kurtosis,-0.264,-0.246,1.000


절대 상관계수 0.85 이상의 심한 중복은 없습니다. Feature Set B를 CV 비교 후보로 유지합니다.


### 3.2 Feature Set C 탐색 변수 점검

Feature Set C의 온도, 충전시간, C-rate는 초기 100사이클 안에서 알 수 있어 데이터 누수는 아니다. 다만 DAY1에서 배치별 관계가 안정적이지 않았으므로 핵심변수가 아니라 비교군으로만 둔다.

이 단계에서는 Batch 1 Train 안의 predictor 간 상관과 결측치만 확인한다. Feature Set A보다 Group CV 성능이 안정적으로 좋아지지 않으면 채택하지 않는다.

In [12]:
set_c_train = X_train_all[features_c].copy()
set_c_pearson = set_c_train.corr(method="pearson")
set_c_spearman = set_c_train.corr(method="spearman")

print("Feature Set C - Pearson correlation (Batch 1 Train only)")
display(set_c_pearson.round(3))

print("Feature Set C - Spearman correlation (Batch 1 Train only)")
display(set_c_spearman.round(3))

set_missingness = pd.DataFrame({
    set_name: X_train_all[columns].isna().sum()
    for set_name, columns in FEATURE_SETS.items()
}).fillna(0).astype(int)

print("Feature Set별 Train 결측치 수")
display(set_missingness.loc[set_missingness.sum(axis=1).gt(0)])

Feature Set C - Pearson correlation (Batch 1 Train only)


,deltaQ_logvar,Tavg_mean_2_100,charge_time_mean_2_100,c_rate_1
deltaQ_logvar,1.000,0.434,-0.608,0.544
Tavg_mean_2_100,0.434,1.000,-0.184,0.550
charge_time_mean_2_100,-0.608,-0.184,1.000,-0.191
c_rate_1,0.544,0.550,-0.191,1.000


Feature Set C - Spearman correlation (Batch 1 Train only)


,deltaQ_logvar,Tavg_mean_2_100,charge_time_mean_2_100,c_rate_1
deltaQ_logvar,1.000,0.280,-0.708,0.480
Tavg_mean_2_100,0.280,1.000,-0.082,0.501
charge_time_mean_2_100,-0.708,-0.082,1.000,-0.067
c_rate_1,0.480,0.501,-0.067,1.000


Feature Set별 Train 결측치 수


,A_deltaQ_logvar,B_deltaQ_shape,C_exploratory_aux


### 3.3 Feature Set별 고정 행렬 준비

Train과 Valid는 후보 선택과 확인에 사용하고, Test는 X만 잠금 상태로 준비한다. y_test는 아직 생성하지 않는다.

In [13]:
X_train_by_set = {
    set_name: train_df[columns].copy()
    for set_name, columns in FEATURE_SETS.items()
}

X_valid_by_set = {
    set_name: valid_df[columns].copy()
    for set_name, columns in FEATURE_SETS.items()
}

X_test_locked_by_set = {
    set_name: test_df[columns].copy()
    for set_name, columns in FEATURE_SETS.items()
}

for set_name, columns in FEATURE_SETS.items():
    assert X_train_by_set[set_name].columns.tolist() == columns
    assert X_valid_by_set[set_name].columns.tolist() == columns
    assert X_test_locked_by_set[set_name].columns.tolist() == columns
    assert TARGET_COLUMN not in X_train_by_set[set_name].columns
    assert not (set(columns) & EXCLUDED_COLUMNS)

feature_matrix_summary = pd.DataFrame([
    {
        "feature_set": set_name,
        "train_shape": X_train_by_set[set_name].shape,
        "valid_shape": X_valid_by_set[set_name].shape,
        "test_locked_shape": X_test_locked_by_set[set_name].shape,
        "test_target_created": False,
    }
    for set_name in FEATURE_SETS
])

display(feature_matrix_summary)
print("Feature Set A/B/C 고정 완료")
print("Batch 2의 y_test와 성능은 아직 확인하지 않음")

,feature_set,train_shape,valid_shape,test_locked_shape,test_target_created
0,A_deltaQ_logvar,"(35, 1)","(11, 1)","(39, 1)",False
1,B_deltaQ_shape,"(35, 3)","(11, 3)","(39, 3)",False
2,C_exploratory_aux,"(35, 4)","(11, 4)","(39, 4)",False


Feature Set A/B/C 고정 완료
Batch 2의 y_test와 성능은 아직 확인하지 않음


### 3.4 선택 규칙

1. Feature Set A를 가장 단순한 기준으로 둔다.
2. B와 C는 같은 모델·같은 GroupKFold·같은 평가 지표로 A와 비교한다.
3. 평균 MAPE뿐 아니라 fold 표준편차와 Train-Valid Gap도 함께 본다.
4. 성능 차이가 작거나 불안정하면 더 단순한 Feature Set A를 선택한다.
5. Batch 2 Test 결과는 Feature Set 선택에 사용하지 않는다.

## 4. 후보 모델 사전 정의

후보 모델과 주요 하이퍼파라미터를 Batch 2 Test 평가 전에 고정한다. 목적은 가장 복잡한 모델을 찾는 것이 아니라 DAY1에서 확인한 데이터 특성과 각 모델의 장단점을 비교하는 것이다.

1. **Median Baseline**: 복잡한 모델이 단순 중앙값 예측보다 실제로 개선되는지 확인
2. **Linear Regression**: deltaQ_logvar와 수명의 단조·선형 관계를 가장 직접적으로 설명
3. **Ridge / Elastic Net**: B·C처럼 변수가 여러 개일 때 계수 불안정과 과적합을 완화
4. **Random Forest / Gradient Boosting**: 비선형성과 상호작용의 추가 효과 확인

Feature Set A는 변수가 하나뿐이므로 Ridge·Elastic Net의 다중공선성 완화 장점이 거의 없다. 따라서 규제 선형모델은 Feature Set B·C에만 적용한다.

In [14]:
import sklearn
from sklearn.base import clone, is_regressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import ElasticNet, LinearRegression, Ridge

MODEL_CONFIG_VERSION = "v1"
MODEL_CONFIG_PATH = (
    MODEL_RESULT_DIR / f"candidate_models_{MODEL_CONFIG_VERSION}.json"
)

# JSON으로 저장 가능한 하이퍼파라미터를 단일 원본으로 관리한다.
MODEL_PARAMETER_CONFIG = {
    "Median Baseline": {
        "class_name": "DummyRegressor",
        "params": {
            "strategy": "median",
        },
    },
    "Linear Regression": {
        "class_name": "LinearRegression",
        "params": {},
    },
    "Ridge": {
        "class_name": "Ridge",
        "params": {
            "alpha": 1.0,
        },
    },
    "Elastic Net": {
        "class_name": "ElasticNet",
        "params": {
            "alpha": 0.01,
            "l1_ratio": 0.5,
            "max_iter": 10000,
            "random_state": RANDOM_STATE,
        },
    },
    "Random Forest": {
        "class_name": "RandomForestRegressor",
        "params": {
            "n_estimators": 300,
            "max_depth": 3,
            "min_samples_leaf": 3,
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        },
    },
    "Gradient Boosting": {
        "class_name": "GradientBoostingRegressor",
        "params": {
            "n_estimators": 100,
            "learning_rate": 0.03,
            "max_depth": 2,
            "min_samples_leaf": 3,
            "random_state": RANDOM_STATE,
        },
    },
}

MODEL_CLASSES = {
    "DummyRegressor": DummyRegressor,
    "LinearRegression": LinearRegression,
    "Ridge": Ridge,
    "ElasticNet": ElasticNet,
    "RandomForestRegressor": RandomForestRegressor,
    "GradientBoostingRegressor": GradientBoostingRegressor,
}

models = {
    model_name: MODEL_CLASSES[spec["class_name"]](**spec["params"])
    for model_name, spec in MODEL_PARAMETER_CONFIG.items()
}

print(f"scikit-learn version: {sklearn.__version__}")
print(f"후보 모델 수: {len(models)}")

scikit-learn version: 1.9.1
후보 모델 수: 6


### 4.1 모델과 Feature Set 비교 계획

Median Baseline은 입력변수를 사용하지 않으므로 한 번만 계산한다. Linear Regression과 트리 모델은 A·B·C를 비교하고, Ridge·Elastic Net은 다변수인 B·C에서만 비교한다.

트리 모델은 학습 feature 범위 밖에서도 마지막 leaf의 상수 예측을 반복하므로 선형 추세를 외삽하지 못한다. Batch 2에 Batch 1 최소 수명보다 짧은 셀이 많다는 점을 고려하여 비선형 비교용으로만 둔다.

In [15]:
MODEL_FEATURE_SET_PLAN = {
    "Median Baseline": [
        "A_deltaQ_logvar",
    ],
    "Linear Regression": [
        "A_deltaQ_logvar",
        "B_deltaQ_shape",
        "C_exploratory_aux",
    ],
    "Ridge": [
        "B_deltaQ_shape",
        "C_exploratory_aux",
    ],
    "Elastic Net": [
        "B_deltaQ_shape",
        "C_exploratory_aux",
    ],
    "Random Forest": [
        "A_deltaQ_logvar",
        "B_deltaQ_shape",
        "C_exploratory_aux",
    ],
    "Gradient Boosting": [
        "A_deltaQ_logvar",
        "B_deltaQ_shape",
        "C_exploratory_aux",
    ],
}

MODEL_NOTES = {
    "Median Baseline": {
        "purpose": "최소 성능 기준",
        "extrapolation": "해당 없음; 항상 Train 중앙값 예측",
    },
    "Linear Regression": {
        "purpose": "해석 가능한 선형 기준",
        "extrapolation": "선형 추세 외삽 가능; 정확성은 별도 검증 필요",
    },
    "Ridge": {
        "purpose": "다변수 계수 축소와 안정화",
        "extrapolation": "선형 추세 외삽 가능; alpha에 따라 계수 축소",
    },
    "Elastic Net": {
        "purpose": "다변수 규제와 변수 선택 보조",
        "extrapolation": "선형 추세 외삽 가능; 규제로 계수 축소",
    },
    "Random Forest": {
        "purpose": "비선형성과 상호작용 비교",
        "extrapolation": "학습 feature 범위 밖의 추세 외삽 불가",
    },
    "Gradient Boosting": {
        "purpose": "저복잡도 비선형 보정 비교",
        "extrapolation": "학습 feature 범위 밖의 추세 외삽에 취약",
    },
}

assert set(MODEL_FEATURE_SET_PLAN) == set(models), (
    "후보 모델과 Feature Set 계획의 모델명이 일치하지 않습니다."
)

for model_name, feature_set_names in MODEL_FEATURE_SET_PLAN.items():
    unknown_feature_sets = set(feature_set_names) - set(FEATURE_SETS)
    assert not unknown_feature_sets, (
        f"{model_name}에 정의되지 않은 Feature Set이 있습니다: "
        f"{sorted(unknown_feature_sets)}"
    )

comparison_plan = pd.DataFrame([
    {
        "model": model_name,
        "feature_set": feature_set_name,
        "n_features": len(FEATURE_SETS[feature_set_name]),
        "purpose": MODEL_NOTES[model_name]["purpose"],
        "extrapolation": MODEL_NOTES[model_name]["extrapolation"],
    }
    for model_name, feature_set_names in MODEL_FEATURE_SET_PLAN.items()
    for feature_set_name in feature_set_names
])

display(comparison_plan)
print(f"예정된 모델-Feature Set 조합 수: {len(comparison_plan)}")

,model,feature_set,n_features,purpose,extrapolation
0,Median Baseline,A_deltaQ_logvar,1,최소 성능 기준,해당 없음; 항상 Train 중앙값 예측
1,Linear Regression,A_deltaQ_logvar,1,해석 가능한 선형 기준,선형 추세 외삽 가능; 정확성은 별도 검증 필요
2,Linear Regression,B_deltaQ_shape,3,해석 가능한 선형 기준,선형 추세 외삽 가능; 정확성은 별도 검증 필요
3,Linear Regression,C_exploratory_aux,4,해석 가능한 선형 기준,선형 추세 외삽 가능; 정확성은 별도 검증 필요
4,Ridge,B_deltaQ_shape,3,다변수 계수 축소와 안정화,선형 추세 외삽 가능; alpha에 따라 계수 축소
5,Ridge,C_exploratory_aux,4,다변수 계수 축소와 안정화,선형 추세 외삽 가능; alpha에 따라 계수 축소
6,Elastic Net,B_deltaQ_shape,3,다변수 규제와 변수 선택 보조,선형 추세 외삽 가능; 규제로 계수 축소
7,Elastic Net,C_exploratory_aux,4,다변수 규제와 변수 선택 보조,선형 추세 외삽 가능; 규제로 계수 축소
8,Random Forest,A_deltaQ_logvar,1,비선형성과 상호작용 비교,학습 feature 범위 밖의 추세 외삽 불가
9,Random Forest,B_deltaQ_shape,3,비선형성과 상호작용 비교,학습 feature 범위 밖의 추세 외삽 불가


예정된 모델-Feature Set 조합 수: 14


### 4.2 후보 모델 안전검사와 설정 고정

모든 객체가 회귀모델이며 복제 가능한지 확인한다. 표본이 작으므로 트리 깊이와 leaf 최소 표본 수를 제한한다. 후보 설정은 JSON으로 저장하고, 이후 실행에서 코드와 저장 설정이 다르면 중단한다.

향후 Train Group CV만 사용한 하이퍼파라미터 탐색이 필요하면 기존 v1을 덮어쓰지 않고 v2 설정을 별도로 만든다. Batch 2 결과를 본 뒤 설정 버전을 추가해서는 안 된다.

In [16]:
for model_name, model in models.items():
    assert is_regressor(model), f"{model_name}은 회귀모델이 아닙니다."
    clone(model)

rf_model = models["Random Forest"]
gb_model = models["Gradient Boosting"]

assert rf_model.max_depth <= 3
assert rf_model.min_samples_leaf >= 3
assert gb_model.max_depth <= 2
assert gb_model.min_samples_leaf >= 3
assert models["Elastic Net"].max_iter >= 10000

model_candidate_config = {
    "version": MODEL_CONFIG_VERSION,
    "random_state": RANDOM_STATE,
    "selection_data": "Batch 1 Train Group CV only",
    "test_data_used_for_selection": False,
    "sklearn_version": sklearn.__version__,
    "models": MODEL_PARAMETER_CONFIG,
    "model_feature_set_plan": MODEL_FEATURE_SET_PLAN,
}

if MODEL_CONFIG_PATH.exists():
    with MODEL_CONFIG_PATH.open("r", encoding="utf-8") as file:
        saved_model_candidate_config = json.load(file)
    assert saved_model_candidate_config == model_candidate_config, (
        "저장된 후보 모델 설정과 현재 코드가 다릅니다. "
        "Test 결과 확인 후 모델 설정을 변경하지 않았는지 확인하세요."
    )
    print("기존 후보 모델 manifest를 재사용합니다.")
else:
    with MODEL_CONFIG_PATH.open("w", encoding="utf-8") as file:
        json.dump(
            model_candidate_config,
            file,
            ensure_ascii=False,
            indent=2,
        )
    print("후보 모델 manifest를 최초 생성했습니다.")

model_parameter_table = pd.DataFrame([
    {
        "model": model_name,
        "class": spec["class_name"],
        "parameters": json.dumps(
            spec["params"],
            ensure_ascii=False,
            sort_keys=True,
        ),
        "feature_sets": ", ".join(
            MODEL_FEATURE_SET_PLAN[model_name]
        ),
    }
    for model_name, spec in MODEL_PARAMETER_CONFIG.items()
])

display(model_parameter_table)
print(f"후보 모델 설정 고정 경로: {MODEL_CONFIG_PATH}")
print("아직 어떤 모델도 fit하지 않았고 Batch 2 성능도 확인하지 않았습니다.")

기존 후보 모델 manifest를 재사용합니다.


,model,class,parameters,feature_sets
0,Median Baseline,DummyRegressor,"{""strategy"": ""median""}",A_deltaQ_logvar
1,Linear Regression,LinearRegression,{},"A_deltaQ_logvar, B_deltaQ_shape, C_exploratory..."
2,Ridge,Ridge,"{""alpha"": 1.0}","B_deltaQ_shape, C_exploratory_aux"
3,Elastic Net,ElasticNet,"{""alpha"": 0.01, ""l1_ratio"": 0.5, ""max_iter"": 1...","B_deltaQ_shape, C_exploratory_aux"
4,Random Forest,RandomForestRegressor,"{""max_depth"": 3, ""min_samples_leaf"": 3, ""n_est...","A_deltaQ_logvar, B_deltaQ_shape, C_exploratory..."
5,Gradient Boosting,GradientBoostingRegressor,"{""learning_rate"": 0.03, ""max_depth"": 2, ""min_s...","A_deltaQ_logvar, B_deltaQ_shape, C_exploratory..."


후보 모델 설정 고정 경로: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/candidate_models_v1.json
아직 어떤 모델도 fit하지 않았고 Batch 2 성능도 확인하지 않았습니다.


### 4.3 비교 시 해석 원칙

1. Median Baseline보다 개선되지 않는 모델은 채택하지 않는다.
2. Feature Set A의 Linear Regression을 가장 중요한 해석 기준으로 둔다.
3. B·C에서는 Ridge·Elastic Net으로 다변수 규제 효과를 확인한다.
4. Random Forest·Gradient Boosting은 낮은 깊이로 비선형 이득만 확인한다.
5. 평균 MAPE 차이가 작으면 표준편차가 작고 설명이 단순한 모델을 선택한다.
6. 트리 모델의 Batch 2 단수명 외삽 한계를 최종 결과 해석에 반드시 기록한다.
7. 아직 후보만 정의한 단계이며 최종 모델은 Batch 1 Group CV와 고정 Valid 결과로 결정한다.

## 5. 전처리·모델 Pipeline 구성

결측치 대체와 표준화는 전체 데이터에 미리 적용하지 않고 각 CV Train fold 안에서만 학습해야 한다. 이를 보장하기 위해 전처리와 모델을 하나의 scikit-learn Pipeline으로 묶는다.

- **공통**: 수치형 X 결측치는 해당 Train fold의 중앙값으로 대체
- **Linear Regression·Ridge·Elastic Net**: 중앙값 대체 후 StandardScaler 적용
- **Median Baseline**: 중앙값 대체만 적용
- **Random Forest·Gradient Boosting**: 중앙값 대체만 적용하고 표준화하지 않음

deltaQ_logvar는 ΔQ 분산에 log10 변환을 적용해 이미 생성된 변수다. 따라서 Pipeline에서 다시 로그 변환하지 않는다.

In [17]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PIPELINE_CONFIG_VERSION = "v1"
PIPELINE_CONFIG_PATH = (
    MODEL_RESULT_DIR / f"pipeline_config_{PIPELINE_CONFIG_VERSION}.json"
)

BASELINE_MODEL_NAMES = {
    "Median Baseline",
}

LINEAR_MODEL_NAMES = {
    "Linear Regression",
    "Ridge",
    "Elastic Net",
}

TREE_MODEL_NAMES = {
    "Random Forest",
    "Gradient Boosting",
}

assert (
    BASELINE_MODEL_NAMES
    | LINEAR_MODEL_NAMES
    | TREE_MODEL_NAMES
) == set(models), "Pipeline 유형에 배정되지 않은 후보 모델이 있습니다."


def build_model_pipeline(model_name, model):
    steps = [
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ]

    if model_name in LINEAR_MODEL_NAMES:
        steps.append(
            (
                "scaler",
                StandardScaler(),
            )
        )

    steps.append(
        (
            "model",
            clone(model),
        )
    )
    return Pipeline(steps=steps)


pipelines = {
    model_name: build_model_pipeline(model_name, model)
    for model_name, model in models.items()
}

print(f"생성된 Pipeline 수: {len(pipelines)}")

생성된 Pipeline 수: 6


### 5.1 Pipeline 구성 검사

선형계열에만 StandardScaler가 포함됐는지, 모든 모델 앞에 중앙값 SimpleImputer가 있는지 확인한다. 전처리기가 아직 fit되지 않은 상태인지도 검사하여 전체 데이터에 미리 학습된 전처리기를 사용하지 않도록 한다.

In [18]:
pipeline_audit_rows = []

for model_name, pipeline in pipelines.items():
    assert isinstance(pipeline, Pipeline)
    assert list(pipeline.named_steps)[0] == "imputer"
    assert list(pipeline.named_steps)[-1] == "model"

    imputer = pipeline.named_steps["imputer"]
    assert isinstance(imputer, SimpleImputer)
    assert imputer.strategy == "median"
    assert not hasattr(imputer, "statistics_"), (
        f"{model_name} imputer가 CV 전에 fit되어 있습니다."
    )

    has_scaler = "scaler" in pipeline.named_steps
    expected_scaler = model_name in LINEAR_MODEL_NAMES
    assert has_scaler == expected_scaler, (
        f"{model_name}의 scaler 적용 여부가 설계와 다릅니다."
    )

    if has_scaler:
        scaler = pipeline.named_steps["scaler"]
        assert isinstance(scaler, StandardScaler)
        assert not hasattr(scaler, "mean_"), (
            f"{model_name} scaler가 CV 전에 fit되어 있습니다."
        )

    assert pipeline.named_steps["model"].__class__ == models[
        model_name
    ].__class__

    forbidden_log_steps = [
        step_name
        for step_name in pipeline.named_steps
        if "log" in step_name.lower()
    ]
    assert not forbidden_log_steps, (
        f"{model_name}에 추가 로그 변환 단계가 있습니다: "
        f"{forbidden_log_steps}"
    )

    clone(pipeline)

    pipeline_audit_rows.append({
        "model": model_name,
        "steps": " → ".join(pipeline.named_steps),
        "imputer": "median",
        "standard_scaler": has_scaler,
        "additional_log_transform": False,
        "fitted_before_cv": False,
    })

pipeline_audit = pd.DataFrame(pipeline_audit_rows)
display(pipeline_audit)

,model,steps,imputer,standard_scaler,additional_log_transform,fitted_before_cv
0,Median Baseline,imputer → model,median,False,False,False
1,Linear Regression,imputer → scaler → model,median,True,False,False
2,Ridge,imputer → scaler → model,median,True,False,False
3,Elastic Net,imputer → scaler → model,median,True,False,False
4,Random Forest,imputer → model,median,False,False,False
5,Gradient Boosting,imputer → model,median,False,False,False


### 5.2 데이터 누수 방지 확인

Pipeline을 cross_validate에 전달하면 각 fold에서 다음 순서가 독립적으로 수행된다.

1. Fold Train에서만 결측치 중앙값 계산
2. 선형모델은 Fold Train에서만 평균·표준편차 계산
3. Fold Validation에는 Train에서 학습한 값으로 transform만 적용
4. 변환된 Fold Train으로 모델 fit
5. Fold Validation 예측

따라서 전체 Batch 1에 먼저 imputer나 scaler를 fit한 뒤 CV를 수행하면 안 된다.

In [19]:
PRECOMPUTED_TRANSFORMED_FEATURES = {
    "deltaQ_logvar": "log10(abs(var(deltaQ(V)))) - feature 생성 단계에서 완료",
}

for set_name, columns in FEATURE_SETS.items():
    if "deltaQ_logvar" in columns:
        assert "deltaQ_logvar" in PRECOMPUTED_TRANSFORMED_FEATURES

for model_name, feature_set_names in MODEL_FEATURE_SET_PLAN.items():
    for feature_set_name in feature_set_names:
        columns = FEATURE_SETS[feature_set_name]
        train_matrix = X_train_by_set[feature_set_name]
        assert train_matrix.columns.tolist() == columns
        assert not (set(columns) & EXCLUDED_COLUMNS)

print("Pipeline 누수 방지 검사 완료")
print("deltaQ_logvar 추가 로그 변환 없음")
print("Imputer와 Scaler는 아직 fit되지 않음")

Pipeline 누수 방지 검사 완료
deltaQ_logvar 추가 로그 변환 없음
Imputer와 Scaler는 아직 fit되지 않음


### 5.3 Pipeline 설정 고정

모델별 전처리 단계와 deltaQ_logvar의 추가 변환 금지 규칙을 JSON으로 저장한다. 이후 CV와 Hold-out 평가에서는 pipelines에서 필요한 객체를 clone하여 사용한다.

In [20]:
PIPELINE_STEPS_BY_MODEL = {
    model_name: list(pipeline.named_steps)
    for model_name, pipeline in pipelines.items()
}

pipeline_config = {
    "version": PIPELINE_CONFIG_VERSION,
    "sklearn_version": sklearn.__version__,
    "imputer": {
        "class_name": "SimpleImputer",
        "strategy": "median",
        "fit_scope": "each CV train fold only",
    },
    "scaler": {
        "class_name": "StandardScaler",
        "applied_models": sorted(LINEAR_MODEL_NAMES),
        "fit_scope": "each CV train fold only",
    },
    "unscaled_models": sorted(
        BASELINE_MODEL_NAMES | TREE_MODEL_NAMES
    ),
    "pipeline_steps_by_model": PIPELINE_STEPS_BY_MODEL,
    "deltaQ_logvar": {
        "already_log_transformed": True,
        "additional_log_transform": False,
    },
}

if PIPELINE_CONFIG_PATH.exists():
    with PIPELINE_CONFIG_PATH.open("r", encoding="utf-8") as file:
        saved_pipeline_config = json.load(file)
    assert saved_pipeline_config == pipeline_config, (
        "저장된 Pipeline 설정과 현재 코드가 다릅니다. "
        "Test 평가 전후 설정 변경 여부를 확인하세요."
    )
    print("기존 Pipeline manifest를 재사용합니다.")
else:
    with PIPELINE_CONFIG_PATH.open("w", encoding="utf-8") as file:
        json.dump(
            pipeline_config,
            file,
            ensure_ascii=False,
            indent=2,
        )
    print("Pipeline manifest를 최초 생성했습니다.")

print(f"Pipeline 설정 고정 경로: {PIPELINE_CONFIG_PATH}")
print("아직 Pipeline fit과 모델 성능 평가는 수행하지 않았습니다.")

기존 Pipeline manifest를 재사용합니다.
Pipeline 설정 고정 경로: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/pipeline_config_v1.json
아직 Pipeline fit과 모델 성능 평가는 수행하지 않았습니다.


### 5.4 다음 Group CV에서의 사용 방법

다음 평가 단계에서는 모델 객체가 아니라 Pipeline 전체를 cross_validate에 전달한다.

    selected_pipeline = clone(pipelines[model_name])

    cv_result = cross_validate(
        estimator=selected_pipeline,
        X=X_train_by_set[feature_set_name],
        y=y_train,
        groups=groups_train,
        cv=group_cv,
        scoring=scoring,
    )

이 구조에서 결측치 대체와 표준화는 매 fold의 Train 내부에서 다시 fit되므로 Validation 정보가 전처리에 섞이지 않는다.

## 6. Batch 1 Train Policy Group Cross-Validation

고정 Hold-out 11셀과 Batch 2 Test는 사용하지 않고 Batch 1 Train 35셀 안에서만 후보를 비교한다.

- 분할: 5-fold GroupKFold
- Group: policy
- 주 평가 지표: MAPE
- 보조 지표: MAE, RMSE, R²
- 보고 방식: fold 평균 ± 표본 표준편차
- 비교 대상: 사전에 고정한 14개 모델–Feature Set 조합

모든 전처리는 Pipeline 안에 있으므로 각 fold의 Train에서만 중앙값과 표준화 계수가 학습된다.

In [21]:
from sklearn.metrics import (
    make_scorer,
    mean_absolute_percentage_error,
)
from sklearn.model_selection import cross_validate

CV_CONFIG_VERSION = "v1"
CV_CONFIG_PATH = MODEL_RESULT_DIR / f"cv_config_{CV_CONFIG_VERSION}.json"
CV_FOLD_RESULT_PATH = (
    MODEL_RESULT_DIR / f"group_cv_fold_results_{CV_CONFIG_VERSION}.csv"
)
CV_SUMMARY_PATH = (
    MODEL_RESULT_DIR / f"group_cv_summary_{CV_CONFIG_VERSION}.csv"
)

scoring = {
    "MAPE": make_scorer(
        mean_absolute_percentage_error,
        greater_is_better=False,
    ),
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}

cv_config = {
    "version": CV_CONFIG_VERSION,
    "selection_data": "Batch 1 Train only",
    "n_train_cells": len(train_df),
    "group_column": GROUP_COLUMN,
    "n_groups": int(groups_train.nunique()),
    "splitter": "GroupKFold",
    "n_splits": N_GROUP_FOLDS,
    "scoring": {
        "primary": "MAPE",
        "secondary": ["MAE", "RMSE", "R2"],
    },
    "std_ddof": 1,
    "return_train_score": True,
    "valid_holdout_used": False,
    "batch2_test_used": False,
}

if CV_CONFIG_PATH.exists():
    with CV_CONFIG_PATH.open("r", encoding="utf-8") as file:
        saved_cv_config = json.load(file)
    assert saved_cv_config == cv_config, (
        "저장된 CV 설정과 현재 코드가 다릅니다."
    )
    print("기존 CV 설정 manifest를 재사용합니다.")
else:
    with CV_CONFIG_PATH.open("w", encoding="utf-8") as file:
        json.dump(cv_config, file, ensure_ascii=False, indent=2)
    print("CV 설정 manifest를 최초 생성했습니다.")

print(f"CV 설정 고정 경로: {CV_CONFIG_PATH}")
print(f"Scoring metrics: {list(scoring)}")

기존 CV 설정 manifest를 재사용합니다.
CV 설정 고정 경로: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/cv_config_v1.json
Scoring metrics: ['MAPE', 'MAE', 'RMSE', 'R2']


### 6.1 14개 후보 조합 평가

scikit-learn의 오차 scorer는 큰 값이 좋은 규칙을 맞추기 위해 음수로 반환된다. 저장·보고할 때 MAPE, MAE, RMSE에 -1을 곱하고 MAPE는 100을 곱해 백분율로 변환한다.

각 모델은 동일한 GroupKFold와 동일한 fold를 사용한다. 따라서 모델 간 성능 차이가 분할 차이에서 발생하지 않는다.

In [22]:
fold_result_rows = []
summary_rows = []

for plan_row in comparison_plan.itertuples(index=False):
    model_name = plan_row.model
    feature_set_name = plan_row.feature_set

    selected_pipeline = clone(pipelines[model_name])
    selected_X_train = X_train_by_set[feature_set_name]

    cv_result = cross_validate(
        estimator=selected_pipeline,
        X=selected_X_train,
        y=y_train,
        groups=groups_train,
        cv=group_cv,
        scoring=scoring,
        return_train_score=True,
        error_score="raise",
        n_jobs=1,
    )

    train_mape_pct = -np.asarray(cv_result["train_MAPE"]) * 100
    cv_mape_pct = -np.asarray(cv_result["test_MAPE"]) * 100
    train_mae = -np.asarray(cv_result["train_MAE"])
    cv_mae = -np.asarray(cv_result["test_MAE"])
    train_rmse = -np.asarray(cv_result["train_RMSE"])
    cv_rmse = -np.asarray(cv_result["test_RMSE"])
    train_r2 = np.asarray(cv_result["train_R2"])
    cv_r2 = np.asarray(cv_result["test_R2"])

    for fold_index in range(N_GROUP_FOLDS):
        fold_result_rows.append({
            "model": model_name,
            "feature_set": feature_set_name,
            "fold": fold_index + 1,
            "train_mape_pct": train_mape_pct[fold_index],
            "cv_mape_pct": cv_mape_pct[fold_index],
            "train_mae": train_mae[fold_index],
            "cv_mae": cv_mae[fold_index],
            "train_rmse": train_rmse[fold_index],
            "cv_rmse": cv_rmse[fold_index],
            "train_r2": train_r2[fold_index],
            "cv_r2": cv_r2[fold_index],
            "fit_time_sec": cv_result["fit_time"][fold_index],
            "score_time_sec": cv_result["score_time"][fold_index],
        })

    summary_rows.append({
        "model": model_name,
        "feature_set": feature_set_name,
        "n_features": len(FEATURE_SETS[feature_set_name]),
        "train_mape_mean_pct": train_mape_pct.mean(),
        "train_mape_std_pct": train_mape_pct.std(ddof=1),
        "cv_mape_mean_pct": cv_mape_pct.mean(),
        "cv_mape_std_pct": cv_mape_pct.std(ddof=1),
        "gap_train_cv_pp": cv_mape_pct.mean() - train_mape_pct.mean(),
        "cv_mae_mean": cv_mae.mean(),
        "cv_mae_std": cv_mae.std(ddof=1),
        "cv_rmse_mean": cv_rmse.mean(),
        "cv_rmse_std": cv_rmse.std(ddof=1),
        "cv_r2_mean": cv_r2.mean(),
        "cv_r2_std": cv_r2.std(ddof=1),
        "fit_time_mean_sec": np.asarray(cv_result["fit_time"]).mean(),
    })

cv_fold_results = pd.DataFrame(fold_result_rows)
cv_summary = pd.DataFrame(summary_rows)

baseline_cv_mape = cv_summary.loc[
    cv_summary["model"].eq("Median Baseline"),
    "cv_mape_mean_pct",
].iloc[0]

cv_summary["improvement_vs_baseline_pp"] = (
    baseline_cv_mape - cv_summary["cv_mape_mean_pct"]
)
cv_summary["relative_improvement_vs_baseline_pct"] = (
    cv_summary["improvement_vs_baseline_pp"]
    / baseline_cv_mape
    * 100
)

cv_summary = cv_summary.sort_values(
    ["cv_mape_mean_pct", "cv_mape_std_pct", "n_features"],
    ascending=[True, True, True],
).reset_index(drop=True)
cv_summary.insert(0, "cv_rank", np.arange(1, len(cv_summary) + 1))

cv_fold_results.to_csv(CV_FOLD_RESULT_PATH, index=False)
cv_summary.to_csv(CV_SUMMARY_PATH, index=False)

print(f"평가한 조합 수: {len(cv_summary)}")
print(f"저장한 fold 결과 수: {len(cv_fold_results)}")
print(f"Fold 결과 저장: {CV_FOLD_RESULT_PATH}")
print(f"요약 결과 저장: {CV_SUMMARY_PATH}")

평가한 조합 수: 14
저장한 fold 결과 수: 70
Fold 결과 저장: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/group_cv_fold_results_v1.csv
요약 결과 저장: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/group_cv_summary_v1.csv


### 6.2 Group CV 결과 보고

순위는 Batch 1 Train의 평균 CV MAPE가 낮은 순서로 정렬한다. 같은 수준이면 fold 표준편차와 변수 수를 함께 보고 더 안정적이고 단순한 후보를 우선한다.

여기서 gap_train_cv_pp는 CV MAPE − Train MAPE다. 양수가 클수록 Train에 비해 보지 못한 policy에서 오차가 커진다는 뜻이다. 과제의 최종 Gap(Train–Valid)은 이후 고정 Hold-out 평가에서 별도로 계산한다.

In [23]:
report_columns = [
    "cv_rank",
    "model",
    "feature_set",
    "n_features",
    "train_mape_mean_pct",
    "cv_mape_mean_pct",
    "cv_mape_std_pct",
    "gap_train_cv_pp",
    "cv_mae_mean",
    "cv_rmse_mean",
    "cv_r2_mean",
    "improvement_vs_baseline_pp",
]

cv_report = cv_summary[report_columns].copy()
numeric_report_columns = cv_report.select_dtypes(include=[np.number]).columns
cv_report[numeric_report_columns] = cv_report[numeric_report_columns].round(3)

display(cv_report)

cv_leader = cv_summary.iloc[0]

print(
    f"CV 선두 후보: {cv_leader['model']} / "
    f"{cv_leader['feature_set']}"
)
print(
    "Batch 1 Group CV MAPE: "
    f"{cv_leader['cv_mape_mean_pct']:.2f}% ± "
    f"{cv_leader['cv_mape_std_pct']:.2f}%"
)
print(
    "Median Baseline 대비 개선: "
    f"{cv_leader['improvement_vs_baseline_pp']:.2f}%p"
)
print("아직 최종 모델이 아니며 고정 Hold-out 확인이 남아 있습니다.")

,cv_rank,model,feature_set,n_features,train_mape_mean_pct,cv_mape_mean_pct,cv_mape_std_pct,gap_train_cv_pp,cv_mae_mean,cv_rmse_mean,cv_r2_mean,improvement_vs_baseline_pp
0,1,Linear Regression,A_deltaQ_logvar,1,7.251,7.598,1.574,0.347,63.482,80.131,0.603,13.530
1,2,Linear Regression,C_exploratory_aux,4,7.051,8.745,1.753,1.694,71.307,91.177,0.468,12.383
2,3,Elastic Net,C_exploratory_aux,4,7.050,8.805,1.749,1.755,71.668,91.407,0.472,12.324
3,4,Elastic Net,B_deltaQ_shape,3,6.752,8.890,2.738,2.138,70.686,84.582,0.540,12.238
4,5,Linear Regression,B_deltaQ_shape,3,6.765,8.892,2.757,2.127,70.714,84.647,0.540,12.237
5,6,Ridge,B_deltaQ_shape,3,6.722,8.896,2.692,2.174,70.604,84.952,0.532,12.232
6,7,Ridge,C_exploratory_aux,4,7.095,9.233,1.923,2.138,74.691,93.713,0.465,11.896
7,8,Gradient Boosting,A_deltaQ_logvar,1,5.718,9.421,4.147,3.703,78.247,106.202,0.350,11.707
8,9,Random Forest,A_deltaQ_logvar,1,6.577,9.658,3.848,3.081,79.910,105.809,0.343,11.470
9,10,Random Forest,B_deltaQ_shape,3,6.293,9.763,3.701,3.469,81.638,109.446,0.327,11.365


CV 선두 후보: Linear Regression / A_deltaQ_logvar
Batch 1 Group CV MAPE: 7.60% ± 1.57%
Median Baseline 대비 개선: 13.53%p
아직 최종 모델이 아니며 고정 Hold-out 확인이 남아 있습니다.


### 6.3 실행 후 누수 상태 확인

cross_validate는 전달된 Pipeline을 fold마다 복제해 학습한다. 원본 pipelines 객체가 fit되지 않았는지 다시 확인한다. Valid Hold-out과 Batch 2 Test의 목표값은 이번 비교에 사용하지 않았다.

In [24]:
for model_name, pipeline in pipelines.items():
    assert not hasattr(
        pipeline.named_steps["imputer"],
        "statistics_",
    ), f"원본 {model_name} imputer가 fit되었습니다."

    if "scaler" in pipeline.named_steps:
        assert not hasattr(
            pipeline.named_steps["scaler"],
            "mean_",
        ), f"원본 {model_name} scaler가 fit되었습니다."

assert cv_config["valid_holdout_used"] is False
assert cv_config["batch2_test_used"] is False
assert len(cv_summary) == len(comparison_plan)
assert len(cv_fold_results) == len(comparison_plan) * N_GROUP_FOLDS
assert cv_fold_results[
    [
        "train_mape_pct",
        "cv_mape_pct",
        "train_mae",
        "cv_mae",
        "train_rmse",
        "cv_rmse",
    ]
].ge(0).all().all()

print("Group CV 사후 누수검사 통과")
print("원본 Pipeline 객체는 미학습 상태")
print("고정 Valid Hold-out 미사용")
print("Batch 2 Test 미사용")

Group CV 사후 누수검사 통과
원본 Pipeline 객체는 미학습 상태
고정 Valid Hold-out 미사용
Batch 2 Test 미사용


### 6.4 다음 단계

CV 평균만 보고 곧바로 최종 모델을 확정하지 않는다. 다음 단계에서는 다음 기준으로 소수 후보를 선정한다.

1. 평균 CV MAPE
2. Fold 간 MAPE 표준편차
3. Train–CV Gap
4. Median Baseline 대비 개선
5. Feature 수와 해석 가능성
6. Batch 2 단수명 구간에 대한 외삽 가능성

선정된 후보만 고정 Batch 1 Hold-out에서 한 번 확인한다.

## 7. Batch 1 CV 결과만으로 최종 후보 잠금

모델과 Feature Set은 **Batch 1 Train Group CV 결과만으로 먼저 확정**한다. 이 단계에서는 Batch 1 Hold-out과 Batch 2 Test를 보지 않는다.

선택 우선순위:

1. Batch 1 Group CV 평균 MAPE
2. CV MAPE 표준편차
3. Median Baseline 대비 개선
4. 해석 가능성
5. 외삽 가능 여부

CV MAPE가 최저 후보와 1%p 이내라면 변수 수가 적고 해석이 단순한 모델을 우선한다. 선택 결과를 manifest로 저장한 뒤에만 Section 8의 Hold-out을 연다.

In [25]:
SELECTION_VERSION = "v1"
SIMPLICITY_TOLERANCE_PP = 1.0

PRE_HOLDOUT_SELECTION_PATH = (
    MODEL_RESULT_DIR / f"pre_holdout_selection_{SELECTION_VERSION}.json"
)
HOLDOUT_CONFIRMATION_PATH = (
    MODEL_RESULT_DIR / f"holdout_confirmation_{SELECTION_VERSION}.json"
)
HOLDOUT_RESULT_PATH = (
    MODEL_RESULT_DIR / f"batch1_holdout_results_{SELECTION_VERSION}.csv"
)
HOLDOUT_PREDICTION_PATH = (
    MODEL_RESULT_DIR / f"batch1_valid_predictions_{SELECTION_VERSION}.csv"
)
SELECTION_CANDIDATE_PATH = (
    MODEL_RESULT_DIR / f"selection_candidates_{SELECTION_VERSION}.csv"
)

MODEL_COMPLEXITY_ORDER = {
    "Linear Regression": 1,
    "Ridge": 2,
    "Elastic Net": 3,
    "Random Forest": 4,
    "Gradient Boosting": 5,
}

selection_candidates = cv_summary[
    ~cv_summary["model"].eq("Median Baseline")
].copy()

best_cv_mape = selection_candidates["cv_mape_mean_pct"].min()
selection_candidates["delta_from_cv_best_pp"] = (
    selection_candidates["cv_mape_mean_pct"] - best_cv_mape
)
selection_candidates["within_1pp_of_best"] = (
    selection_candidates["delta_from_cv_best_pp"]
    <= SIMPLICITY_TOLERANCE_PP
)
selection_candidates["model_complexity_order"] = (
    selection_candidates["model"].map(MODEL_COMPLEXITY_ORDER)
)

cv_shortlist = selection_candidates[
    selection_candidates["within_1pp_of_best"]
].copy()
cv_shortlist = cv_shortlist.sort_values(
    [
        "n_features",
        "model_complexity_order",
        "cv_mape_std_pct",
        "cv_mape_mean_pct",
    ],
    ascending=[True, True, True, True],
)

# 여러 후보가 남으면 Hold-out을 열기 전에 선택 규칙을 먼저 추가한다.
assert len(cv_shortlist) == 1, (
    "1%p 단순성 규칙으로 후보가 하나로 좁혀지지 않았습니다. "
    "Hold-out을 보기 전에 추가 선택 규칙을 먼저 고정하세요."
)

provisional_selection = cv_shortlist.iloc[0]
SELECTED_MODEL_NAME = provisional_selection["model"]
SELECTED_FEATURE_SET_NAME = provisional_selection["feature_set"]
selected_features = FEATURE_SETS[SELECTED_FEATURE_SET_NAME]
selected_cv_mape = float(provisional_selection["cv_mape_mean_pct"])
selected_cv_std = float(provisional_selection["cv_mape_std_pct"])

selection_candidates.to_csv(SELECTION_CANDIDATE_PATH, index=False)

display(
    selection_candidates[
        [
            "cv_rank",
            "model",
            "feature_set",
            "n_features",
            "cv_mape_mean_pct",
            "cv_mape_std_pct",
            "delta_from_cv_best_pp",
            "within_1pp_of_best",
        ]
    ].round(3)
)

print(
    f"잠금 대상: {SELECTED_MODEL_NAME} / "
    f"{SELECTED_FEATURE_SET_NAME}"
)
print(
    f"CV MAPE: {selected_cv_mape:.2f}% ± "
    f"{selected_cv_std:.2f}%"
)
print(f"후보 선정표 저장: {SELECTION_CANDIDATE_PATH}")

,cv_rank,model,feature_set,n_features,cv_mape_mean_pct,cv_mape_std_pct,delta_from_cv_best_pp,within_1pp_of_best
0,1,Linear Regression,A_deltaQ_logvar,1,7.598,1.574,0.000,True
1,2,Linear Regression,C_exploratory_aux,4,8.745,1.753,1.147,False
2,3,Elastic Net,C_exploratory_aux,4,8.805,1.749,1.207,False
3,4,Elastic Net,B_deltaQ_shape,3,8.890,2.738,1.292,False
4,5,Linear Regression,B_deltaQ_shape,3,8.892,2.757,1.294,False
5,6,Ridge,B_deltaQ_shape,3,8.896,2.692,1.298,False
6,7,Ridge,C_exploratory_aux,4,9.233,1.923,1.635,False
7,8,Gradient Boosting,A_deltaQ_logvar,1,9.421,4.147,1.823,False
8,9,Random Forest,A_deltaQ_logvar,1,9.658,3.848,2.061,False
9,10,Random Forest,B_deltaQ_shape,3,9.763,3.701,2.165,False


잠금 대상: Linear Regression / A_deltaQ_logvar
CV MAPE: 7.60% ± 1.57%
후보 선정표 저장: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/selection_candidates_v1.csv


### 7.1 Hold-out 확인 전 최종 후보 잠금

CV로 선택한 모델명, Feature Set, 하이퍼파라미터와 Pipeline을 먼저 JSON으로 저장한다. 이 잠금이 완료된 후에는 Hold-out 결과를 이유로 후보를 바꾸거나 Feature를 추가하지 않는다.

In [26]:
pre_holdout_selection_record = {
    "version": SELECTION_VERSION,
    "selection_data": "Batch 1 Train Group CV only",
    "holdout_used_for_selection": False,
    "batch2_test_used": False,
    "simplicity_tolerance_pp": SIMPLICITY_TOLERANCE_PP,
    "selected_model": SELECTED_MODEL_NAME,
    "selected_feature_set": SELECTED_FEATURE_SET_NAME,
    "selected_features": selected_features,
    "selected_model_parameters": MODEL_PARAMETER_CONFIG[
        SELECTED_MODEL_NAME
    ],
    "pipeline_steps": PIPELINE_STEPS_BY_MODEL[
        SELECTED_MODEL_NAME
    ],
    "cv": {
        "mape_mean_pct": round(selected_cv_mape, 10),
        "mape_std_pct": round(selected_cv_std, 10),
        "train_mape_mean_pct": round(
            float(provisional_selection["train_mape_mean_pct"]),
            10,
        ),
        "gap_train_cv_pp": round(
            float(provisional_selection["gap_train_cv_pp"]),
            10,
        ),
    },
    "decision": (
        "Lowest Group CV MAPE; only non-baseline candidate within "
        "1 percentage point of the CV best; one feature; linear "
        "interpretability and extrapolation capability"
    ),
}

if PRE_HOLDOUT_SELECTION_PATH.exists():
    with PRE_HOLDOUT_SELECTION_PATH.open("r", encoding="utf-8") as file:
        saved_pre_holdout_selection = json.load(file)
    assert saved_pre_holdout_selection == pre_holdout_selection_record, (
        "Hold-out 확인 전 잠근 모델 설계가 기존 기록과 다릅니다."
    )
    print("기존 사전 잠금 manifest를 재사용합니다.")
else:
    with PRE_HOLDOUT_SELECTION_PATH.open("w", encoding="utf-8") as file:
        json.dump(
            pre_holdout_selection_record,
            file,
            ensure_ascii=False,
            indent=2,
        )
    print("Hold-out 확인 전 사전 잠금 manifest를 생성했습니다.")

FINAL_CANDIDATE_LOCKED_BEFORE_HOLDOUT = True
LOCKED_MODEL_NAME = SELECTED_MODEL_NAME
LOCKED_FEATURE_SET_NAME = SELECTED_FEATURE_SET_NAME
LOCKED_FEATURES = list(selected_features)
final_candidate = clone(pipelines[LOCKED_MODEL_NAME])

assert "y_test" not in globals()
assert set(train_df["batch"]) == {"Batch 1"}
assert set(valid_df["batch"]) == {"Batch 1"}
assert pre_holdout_selection_record["holdout_used_for_selection"] is False
assert pre_holdout_selection_record["batch2_test_used"] is False

print("Hold-out 확인 전 최종 후보 잠금 완료")
print(f"선택: {LOCKED_MODEL_NAME} / {LOCKED_FEATURE_SET_NAME}")
print(f"Feature: {LOCKED_FEATURES}")
print(f"사전 잠금 기록: {PRE_HOLDOUT_SELECTION_PATH}")

기존 사전 잠금 manifest를 재사용합니다.
Hold-out 확인 전 최종 후보 잠금 완료
선택: Linear Regression / A_deltaQ_logvar
Feature: ['deltaQ_logvar']
사전 잠금 기록: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/pre_holdout_selection_v1.json


## 8. Batch 1 고정 Hold-out으로 마지막 확인

Section 7에서 **이미 잠근 후보 하나**와 Median Baseline만 고정 Hold-out에서 확인한다. 다른 모델·Feature 조합은 Hold-out에서 비교하지 않는다.

Hold-out은 새로운 policy에 대한 내부 일반화 확인용이다. 결과를 반복해서 보고 모델, Feature 또는 하이퍼파라미터를 변경하면 Hold-out도 사실상 학습 데이터가 되므로, 이 확인은 최종 후보 확정 후 한 번만 수행한다.

In [27]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)


def regression_metrics(y_true, y_pred):
    return {
        "mape_pct": mean_absolute_percentage_error(y_true, y_pred) * 100,
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
    }


assert FINAL_CANDIDATE_LOCKED_BEFORE_HOLDOUT
assert SELECTED_MODEL_NAME == LOCKED_MODEL_NAME
assert SELECTED_FEATURE_SET_NAME == LOCKED_FEATURE_SET_NAME
assert selected_features == LOCKED_FEATURES

# 분석 의사결정 기준상 Hold-out 검토 횟수는 1회로 고정한다.
HOLDOUT_REVIEW_COUNT = 1
assert HOLDOUT_REVIEW_COUNT == 1

final_candidate.fit(
    train_df[selected_features],
    train_df[TARGET_COLUMN],
)
selected_train_prediction = final_candidate.predict(
    train_df[selected_features]
)
valid_pred = final_candidate.predict(
    valid_df[selected_features]
)

selected_train_metrics = regression_metrics(
    train_df[TARGET_COLUMN],
    selected_train_prediction,
)
selected_valid_metrics = regression_metrics(
    valid_df[TARGET_COLUMN],
    valid_pred,
)

baseline_holdout_model = clone(pipelines["Median Baseline"])
baseline_holdout_model.fit(
    train_df[FEATURE_SETS["A_deltaQ_logvar"]],
    train_df[TARGET_COLUMN],
)
baseline_valid_prediction = baseline_holdout_model.predict(
    valid_df[FEATURE_SETS["A_deltaQ_logvar"]]
)
baseline_valid_metrics = regression_metrics(
    valid_df[TARGET_COLUMN],
    baseline_valid_prediction,
)

gap_cv_valid_pp = (
    selected_valid_metrics["mape_pct"] - selected_cv_mape
)
valid_improvement_vs_baseline_pp = (
    baseline_valid_metrics["mape_pct"]
    - selected_valid_metrics["mape_pct"]
)

holdout_results = pd.DataFrame([
    {
        "role": "Median Baseline",
        "model": "Median Baseline",
        "feature_set": "A_deltaQ_logvar",
        "cv_mape_mean_pct": float(
            cv_summary.loc[
                cv_summary["model"].eq("Median Baseline"),
                "cv_mape_mean_pct",
            ].iloc[0]
        ),
        "cv_mape_std_pct": float(
            cv_summary.loc[
                cv_summary["model"].eq("Median Baseline"),
                "cv_mape_std_pct",
            ].iloc[0]
        ),
        "valid_mape_pct": baseline_valid_metrics["mape_pct"],
        "valid_mae": baseline_valid_metrics["mae"],
        "valid_rmse": baseline_valid_metrics["rmse"],
        "valid_r2": baseline_valid_metrics["r2"],
    },
    {
        "role": "Locked Candidate",
        "model": LOCKED_MODEL_NAME,
        "feature_set": LOCKED_FEATURE_SET_NAME,
        "cv_mape_mean_pct": selected_cv_mape,
        "cv_mape_std_pct": selected_cv_std,
        "valid_mape_pct": selected_valid_metrics["mape_pct"],
        "valid_mae": selected_valid_metrics["mae"],
        "valid_rmse": selected_valid_metrics["rmse"],
        "valid_r2": selected_valid_metrics["r2"],
    },
])
holdout_results["gap_cv_valid_pp"] = (
    holdout_results["valid_mape_pct"]
    - holdout_results["cv_mape_mean_pct"]
)

valid_predictions = valid_df[
    ["batch", "cell_id", "policy", TARGET_COLUMN]
].copy()
valid_predictions["prediction"] = valid_pred
valid_predictions["residual_actual_minus_pred"] = (
    valid_predictions[TARGET_COLUMN]
    - valid_predictions["prediction"]
)
valid_predictions["absolute_percentage_error_pct"] = (
    valid_predictions["residual_actual_minus_pred"].abs()
    / valid_predictions[TARGET_COLUMN]
    * 100
)

holdout_results.to_csv(HOLDOUT_RESULT_PATH, index=False)
valid_predictions.to_csv(HOLDOUT_PREDICTION_PATH, index=False)

display(holdout_results.round(3))
display(valid_predictions.round(3))

print(f"Locked Candidate Hold-out MAPE: {selected_valid_metrics['mape_pct']:.2f}%")
print(f"CV–Hold-out Gap: {gap_cv_valid_pp:+.2f}%p")
print(
    "Hold-out Median Baseline 대비 개선: "
    f"{valid_improvement_vs_baseline_pp:.2f}%p"
)
print(f"Hold-out 결과 저장: {HOLDOUT_RESULT_PATH}")
print(f"Hold-out 예측 저장: {HOLDOUT_PREDICTION_PATH}")

,role,model,feature_set,cv_mape_mean_pct,cv_mape_std_pct,valid_mape_pct,valid_mae,valid_rmse,valid_r2,gap_cv_valid_pp
0,Median Baseline,Median Baseline,A_deltaQ_logvar,21.128,8.382,16.114,150.364,192.245,-0.023,-5.014
1,Locked Candidate,Linear Regression,A_deltaQ_logvar,7.598,1.574,11.631,111.954,138.748,0.467,4.034


,batch,cell_id,policy,cycle_life,prediction,residual_actual_minus_pred,absolute_percentage_error_pct
0,Batch 1,b1c0,3_6C-80PER_3_6C,1190.0,1459.531,-269.531,22.650
1,Batch 1,b1c1,3_6C-80PER_3_6C,1179.0,1459.039,-280.039,23.752
2,Batch 1,b1c2,3_6C-80PER_3_6C,1177.0,1307.809,-130.809,11.114
16,Batch 1,b1c16,5_4C-60PER_3_6C,862.0,959.663,-97.663,11.330
17,Batch 1,b1c17,5_4C-60PER_3_6C,857.0,889.085,-32.085,3.744
18,Batch 1,b1c18,5_4C-70PER_3C,691.0,714.310,-23.310,3.373
19,Batch 1,b1c19,5_4C-70PER_3C,788.0,697.834,90.166,11.442
30,Batch 1,b1c30,6C-50PER_3_6C,709.0,765.975,-56.975,8.036
31,Batch 1,b1c31,6C-50PER_3_6C,876.0,784.013,91.987,10.501
34,Batch 1,b1c34,7C-30PER_3_6C,742.0,822.718,-80.718,10.878


Locked Candidate Hold-out MAPE: 11.63%
CV–Hold-out Gap: +4.03%p
Hold-out Median Baseline 대비 개선: 4.48%p
Hold-out 결과 저장: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/batch1_holdout_results_v1.csv
Hold-out 예측 저장: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/batch1_valid_predictions_v1.csv


### 8.1 Hold-out 확인 기록

Hold-out 성능은 모델을 다시 고르는 근거가 아니라 잠긴 후보의 일반화 성능을 보고하기 위한 값으로 저장한다. 선택된 모델과 Feature가 Section 7의 사전 잠금 기록과 같은지도 다시 검사한다.

In [28]:
holdout_confirmation_record = {
    "version": SELECTION_VERSION,
    "selection_locked_before_holdout": True,
    "holdout_review_count": HOLDOUT_REVIEW_COUNT,
    "batch2_test_used": False,
    "selected_model": LOCKED_MODEL_NAME,
    "selected_feature_set": LOCKED_FEATURE_SET_NAME,
    "selected_features": LOCKED_FEATURES,
    "holdout": {
        "mape_pct": round(selected_valid_metrics["mape_pct"], 10),
        "mae": round(selected_valid_metrics["mae"], 10),
        "rmse": round(selected_valid_metrics["rmse"], 10),
        "r2": round(selected_valid_metrics["r2"], 10),
        "gap_cv_valid_pp": round(gap_cv_valid_pp, 10),
        "improvement_vs_baseline_pp": round(
            valid_improvement_vs_baseline_pp,
            10,
        ),
    },
    "post_holdout_model_change_allowed": False,
}

if HOLDOUT_CONFIRMATION_PATH.exists():
    with HOLDOUT_CONFIRMATION_PATH.open("r", encoding="utf-8") as file:
        saved_holdout_confirmation = json.load(file)
    assert saved_holdout_confirmation == holdout_confirmation_record, (
        "기존 Hold-out 확인 기록과 현재 결과가 다릅니다."
    )
    print("기존 Hold-out 확인 기록을 재사용합니다.")
else:
    with HOLDOUT_CONFIRMATION_PATH.open("w", encoding="utf-8") as file:
        json.dump(
            holdout_confirmation_record,
            file,
            ensure_ascii=False,
            indent=2,
        )
    print("Hold-out 확인 기록을 최초 생성했습니다.")

assert holdout_confirmation_record["selected_model"] == (
    pre_holdout_selection_record["selected_model"]
)
assert holdout_confirmation_record["selected_feature_set"] == (
    pre_holdout_selection_record["selected_feature_set"]
)
assert holdout_confirmation_record["post_holdout_model_change_allowed"] is False
assert "y_test" not in globals()

FINAL_SELECTION_LOCKED = True
final_pipeline_for_refit = clone(pipelines[LOCKED_MODEL_NAME])

print("Hold-out 확인 후 모델 변경 없음")
print(f"최종 잠금: {LOCKED_MODEL_NAME} / {LOCKED_FEATURE_SET_NAME}")
print(f"확인 기록: {HOLDOUT_CONFIRMATION_PATH}")
print("Batch 2 Test는 아직 평가하지 않음")

기존 Hold-out 확인 기록을 재사용합니다.
Hold-out 확인 후 모델 변경 없음
최종 잠금: Linear Regression / A_deltaQ_logvar
확인 기록: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/holdout_confirmation_v1.json
Batch 2 Test는 아직 평가하지 않음


### 8.2 Hold-out 결과 해석 원칙

- Hold-out 성능은 새로운 policy에 대한 내부 일반화 확인값으로 보고한다.
- CV보다 Hold-out MAPE가 높으면 그 차이를 CV–Hold-out Gap으로 해석한다.
- Hold-out 결과가 기대보다 낮아도 모델·Feature·하이퍼파라미터를 다시 선택하지 않는다.
- Batch 2의 단수명 영역은 Batch 1 학습범위 밖이므로 Hold-out 성능만으로 외부 일반화를 보장할 수 없다.
- 다음 단계에서는 잠긴 Pipeline을 Batch 1 전체로 다시 fit한 뒤 Batch 2를 한 번만 평가한다.

## 9. 최종 설계 고정 후 Batch 2 단일 평가

Batch 2를 열기 전에 최종 Feature Set, 모델, 하이퍼파라미터, 전처리와 `random_state`를 별도 manifest로 저장한다. 이 시점까지 Batch 2 Target 성능은 평가하지 않았다.

그다음 잠긴 Pipeline을 라벨이 있는 **Batch 1 전체 46셀**로 다시 학습하고, Batch 2 라벨 39셀을 단 한 번 평가한다. Batch 2 결과를 본 뒤 모델·Feature·하이퍼파라미터를 변경하지 않는다.

In [29]:
BATCH2_EVALUATION_VERSION = "v1"
PAPER_TARGET_MAPE_PCT = 9.1

PRE_BATCH2_EVALUATION_PATH = (
    MODEL_RESULT_DIR
    / f"pre_batch2_evaluation_{BATCH2_EVALUATION_VERSION}.json"
)
BATCH2_EVALUATION_RECORD_PATH = (
    MODEL_RESULT_DIR
    / f"batch2_evaluation_record_{BATCH2_EVALUATION_VERSION}.json"
)
BATCH2_TEST_RESULT_PATH = (
    MODEL_RESULT_DIR
    / f"batch2_test_results_{BATCH2_EVALUATION_VERSION}.csv"
)
BATCH2_PREDICTION_PATH = (
    MODEL_RESULT_DIR
    / f"batch2_test_predictions_{BATCH2_EVALUATION_VERSION}.csv"
)
PERFORMANCE_REPORT_PATH = (
    MODEL_RESULT_DIR
    / f"performance_report_{BATCH2_EVALUATION_VERSION}.csv"
)

assert FINAL_SELECTION_LOCKED
assert LOCKED_MODEL_NAME == "Linear Regression"
assert LOCKED_FEATURE_SET_NAME == "A_deltaQ_logvar"
assert LOCKED_FEATURES == ["deltaQ_logvar"]
assert "y_test" not in globals(), (
    "Batch 2 평가 전에 이미 y_test가 생성되었습니다."
)

final_preprocessing = {
    "missing_values": "SimpleImputer(strategy='median')",
    "scaling": "StandardScaler()",
    "target_transform": "none",
    "additional_feature_transform": (
        "none; deltaQ_logvar is already log-transformed"
    ),
}

pre_batch2_evaluation_record = {
    "version": BATCH2_EVALUATION_VERSION,
    "final_feature_set": LOCKED_FEATURE_SET_NAME,
    "final_features": LOCKED_FEATURES,
    "final_model": LOCKED_MODEL_NAME,
    "final_model_parameters": MODEL_PARAMETER_CONFIG[
        LOCKED_MODEL_NAME
    ],
    "pipeline_steps": PIPELINE_STEPS_BY_MODEL[
        LOCKED_MODEL_NAME
    ],
    "preprocessing": final_preprocessing,
    "project_random_state": RANDOM_STATE,
    "model_random_state": (
        "not applicable to deterministic LinearRegression"
    ),
    "refit_data": "all labeled Batch 1 cells",
    "refit_cell_count": int(len(batch1_df)),
    "test_data": "labeled Batch 2 cells",
    "test_cell_count": int(len(test_df)),
    "batch2_evaluated_at_lock_time": False,
    "post_test_model_change_allowed": False,
}

if PRE_BATCH2_EVALUATION_PATH.exists():
    with PRE_BATCH2_EVALUATION_PATH.open(
        "r", encoding="utf-8"
    ) as file:
        saved_pre_batch2_record = json.load(file)
    assert saved_pre_batch2_record == pre_batch2_evaluation_record, (
        "저장된 Batch 2 평가 전 설계와 현재 설계가 다릅니다."
    )
    print("기존 Batch 2 평가 전 manifest를 재사용합니다.")
else:
    with PRE_BATCH2_EVALUATION_PATH.open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(
            pre_batch2_evaluation_record,
            file,
            ensure_ascii=False,
            indent=2,
        )
    print("Batch 2 평가 전 manifest를 생성했습니다.")

BATCH2_DESIGN_LOCKED = True

print("Batch 2 평가 전 최종 설계 고정 완료")
print(f"Feature Set: {LOCKED_FEATURE_SET_NAME}")
print(f"Feature: {LOCKED_FEATURES}")
print(f"Model: {LOCKED_MODEL_NAME}")
print(f"Hyperparameters: {MODEL_PARAMETER_CONFIG[LOCKED_MODEL_NAME]}")
print(f"Preprocessing: {final_preprocessing}")
print(f"Project random_state: {RANDOM_STATE}")
print("Batch 2 evaluated: False")
print(f"사전 기록: {PRE_BATCH2_EVALUATION_PATH}")

기존 Batch 2 평가 전 manifest를 재사용합니다.
Batch 2 평가 전 최종 설계 고정 완료
Feature Set: A_deltaQ_logvar
Feature: ['deltaQ_logvar']
Model: Linear Regression
Hyperparameters: {'class_name': 'LinearRegression', 'params': {}}
Preprocessing: {'missing_values': "SimpleImputer(strategy='median')", 'scaling': 'StandardScaler()', 'target_transform': 'none', 'additional_feature_transform': 'none; deltaQ_logvar is already log-transformed'}
Project random_state: 42
Batch 2 evaluated: False
사전 기록: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/pre_batch2_evaluation_v1.json


### 9.1 Batch 1 전체 재학습 및 Batch 2 1회 평가

Hold-out 분할을 다시 합친 Batch 1 전체로 동일한 Pipeline을 재학습한다. Batch 2 평가 기록이 이미 존재하면 저장된 결과를 불러오며 `predict`와 성능 계산을 반복하지 않는다.

In [30]:
from datetime import datetime, timezone


assert BATCH2_DESIGN_LOCKED
assert pre_batch2_evaluation_record[
    "batch2_evaluated_at_lock_time"
] is False
assert pre_batch2_evaluation_record[
    "post_test_model_change_allowed"
] is False

final_model = clone(final_pipeline_for_refit)
final_model.fit(
    batch1_df[LOCKED_FEATURES],
    batch1_df[TARGET_COLUMN],
)

if BATCH2_EVALUATION_RECORD_PATH.exists():
    required_saved_paths = [
        BATCH2_TEST_RESULT_PATH,
        BATCH2_PREDICTION_PATH,
        PERFORMANCE_REPORT_PATH,
    ]
    assert all(path.exists() for path in required_saved_paths), (
        "Batch 2 평가 기록은 있지만 결과 파일 일부가 없습니다."
    )

    with BATCH2_EVALUATION_RECORD_PATH.open(
        "r", encoding="utf-8"
    ) as file:
        batch2_evaluation_record = json.load(file)

    assert batch2_evaluation_record["final_model"] == LOCKED_MODEL_NAME
    assert batch2_evaluation_record["final_feature_set"] == (
        LOCKED_FEATURE_SET_NAME
    )
    assert batch2_evaluation_record["final_features"] == LOCKED_FEATURES
    assert batch2_evaluation_record[
        "post_test_model_change_allowed"
    ] is False

    batch2_test_results = pd.read_csv(BATCH2_TEST_RESULT_PATH)
    batch2_predictions = pd.read_csv(BATCH2_PREDICTION_PATH)
    performance_report = pd.read_csv(PERFORMANCE_REPORT_PATH)

    test_mape = float(batch2_test_results.loc[0, "test_mape_pct"])
    test_mae = float(batch2_test_results.loc[0, "test_mae"])
    test_rmse = float(batch2_test_results.loc[0, "test_rmse"])
    test_r2 = float(batch2_test_results.loc[0, "test_r2"])
    gap_valid_test_pp = float(
        batch2_test_results.loc[0, "gap_valid_test_pp"]
    )
    gap_target_test_pp = float(
        batch2_test_results.loc[0, "gap_target_test_pp"]
    )
    batch2_evaluation_mode = "loaded_saved_single_evaluation"
    print("저장된 Batch 2 단일 평가 결과를 불러왔습니다.")
else:
    # 이 분기에서만 Batch 2 Target을 열고 한 번 평가한다.
    test_pred = final_model.predict(test_df[LOCKED_FEATURES])
    y_test = test_df[TARGET_COLUMN].copy()

    test_mape = mean_absolute_percentage_error(
        y_test,
        test_pred,
    ) * 100
    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_r2 = r2_score(y_test, test_pred)

    gap_valid_test_pp = (
        test_mape - selected_valid_metrics["mape_pct"]
    )
    gap_target_test_pp = test_mape - PAPER_TARGET_MAPE_PCT

    batch2_predictions = test_df[
        ["batch", "cell_id", "policy", TARGET_COLUMN]
    ].copy()
    batch2_predictions["prediction"] = test_pred
    batch2_predictions["residual_actual_minus_pred"] = (
        batch2_predictions[TARGET_COLUMN]
        - batch2_predictions["prediction"]
    )
    batch2_predictions["absolute_percentage_error_pct"] = (
        batch2_predictions["residual_actual_minus_pred"].abs()
        / batch2_predictions[TARGET_COLUMN]
        * 100
    )

    batch2_test_results = pd.DataFrame([
        {
            "model": LOCKED_MODEL_NAME,
            "feature_set": LOCKED_FEATURE_SET_NAME,
            "n_train_batch1": len(batch1_df),
            "n_test_batch2": len(test_df),
            "test_mape_pct": test_mape,
            "test_mae": test_mae,
            "test_rmse": test_rmse,
            "test_r2": test_r2,
            "gap_valid_test_pp": gap_valid_test_pp,
            "paper_target_mape_pct": PAPER_TARGET_MAPE_PCT,
            "gap_target_test_pp": gap_target_test_pp,
        }
    ])

    performance_report = pd.DataFrame([
        {
            "index": "Train (Batch 1 Group CV)",
            "mape_pct": selected_cv_mape,
            "note": f"mean ± std = {selected_cv_mape:.2f}% ± {selected_cv_std:.2f}%",
        },
        {
            "index": "Valid (Batch 1 Hold-out)",
            "mape_pct": selected_valid_metrics["mape_pct"],
            "note": "locked candidate internal generalization",
        },
        {
            "index": "Test (Batch 2)",
            "mape_pct": test_mape,
            "note": "single external batch evaluation",
        },
        {
            "index": "Gap (Train-Valid)",
            "mape_pct": (
                selected_valid_metrics["mape_pct"] - selected_cv_mape
            ),
            "note": "positive means Hold-out degradation",
        },
        {
            "index": "Gap (Valid-Test)",
            "mape_pct": gap_valid_test_pp,
            "note": "positive means batch generalization degradation",
        },
        {
            "index": "Gap (Target-Test)",
            "mape_pct": gap_target_test_pp,
            "note": f"paper target = {PAPER_TARGET_MAPE_PCT:.1f}%",
        },
    ])

    batch2_test_results.to_csv(BATCH2_TEST_RESULT_PATH, index=False)
    batch2_predictions.to_csv(BATCH2_PREDICTION_PATH, index=False)
    performance_report.to_csv(PERFORMANCE_REPORT_PATH, index=False)

    batch2_evaluation_record = {
        "version": BATCH2_EVALUATION_VERSION,
        "evaluation_count": 1,
        "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
        "training_data": "all 46 labeled Batch 1 cells",
        "test_data": "all 39 labeled Batch 2 cells",
        "final_model": LOCKED_MODEL_NAME,
        "final_feature_set": LOCKED_FEATURE_SET_NAME,
        "final_features": LOCKED_FEATURES,
        "test_mape_pct": round(float(test_mape), 10),
        "test_mae": round(float(test_mae), 10),
        "test_rmse": round(float(test_rmse), 10),
        "test_r2": round(float(test_r2), 10),
        "gap_valid_test_pp": round(float(gap_valid_test_pp), 10),
        "paper_target_mape_pct": PAPER_TARGET_MAPE_PCT,
        "gap_target_test_pp": round(float(gap_target_test_pp), 10),
        "post_test_model_change_allowed": False,
    }
    with BATCH2_EVALUATION_RECORD_PATH.open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(
            batch2_evaluation_record,
            file,
            ensure_ascii=False,
            indent=2,
        )

    batch2_evaluation_mode = "new_single_evaluation"
    print("Batch 2를 최초 1회 평가하고 결과를 저장했습니다.")

assert batch2_evaluation_record["evaluation_count"] == 1
assert batch2_evaluation_record[
    "post_test_model_change_allowed"
] is False

display(batch2_test_results.round(3))
display(performance_report.round(3))

print(f"Evaluation mode: {batch2_evaluation_mode}")
print(f"Batch 2 Test MAPE: {test_mape:.2f}%")
print(f"Batch 2 Test MAE: {test_mae:.2f} cycles")
print(f"Batch 2 Test RMSE: {test_rmse:.2f} cycles")
print(f"Batch 2 Test R²: {test_r2:.3f}")
print(f"Gap (Valid-Test): {gap_valid_test_pp:+.2f}%p")
print(f"Gap (Target-Test): {gap_target_test_pp:+.2f}%p")
print("Batch 2 결과를 이용한 모델·Feature 변경 금지")
print(f"평가 기록: {BATCH2_EVALUATION_RECORD_PATH}")

저장된 Batch 2 단일 평가 결과를 불러왔습니다.


,model,feature_set,n_train_batch1,n_test_batch2,test_mape_pct,test_mae,test_rmse,test_r2,gap_valid_test_pp,paper_target_mape_pct,gap_target_test_pp
0,Linear Regression,A_deltaQ_logvar,46,39,31.529,155.588,169.556,0.402,19.898,9.1,22.429


,index,mape_pct,note
0,Train (Batch 1 Group CV),7.598,mean ± std = 7.60% ± 1.57%
1,Valid (Batch 1 Hold-out),11.631,locked candidate internal generalization
2,Test (Batch 2),31.529,single external batch evaluation
3,Gap (Train-Valid),4.034,positive means Hold-out degradation
4,Gap (Valid-Test),19.898,positive means batch generalization degradation
5,Gap (Target-Test),22.429,paper target = 9.1%


Evaluation mode: loaded_saved_single_evaluation
Batch 2 Test MAPE: 31.53%
Batch 2 Test MAE: 155.59 cycles
Batch 2 Test RMSE: 169.56 cycles
Batch 2 Test R²: 0.402
Gap (Valid-Test): +19.90%p
Gap (Target-Test): +22.43%p
Batch 2 결과를 이용한 모델·Feature 변경 금지
평가 기록: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/batch2_evaluation_record_v1.json


### 9.2 해석 원칙

- Batch 2 MAPE는 잠긴 모델의 외부 배치 일반화 성능이다.
- Valid–Test Gap이 크면 Batch 1 내부 과적합만이 아니라 Batch 1과 Batch 2의 분포 차이와 외삽 난이도를 함께 확인한다.
- 선형회귀는 트리 모델과 달리 학습 Target 범위 밖으로 값을 낼 수 있지만, 외삽값이 정확하다는 뜻은 아니다.
- Batch 2 성능이 원논문 목표인 9.1%보다 낮아도 이 결과를 보고 Feature, 모델 또는 하이퍼파라미터를 다시 변경하지 않는다.
- 개선안은 현재 Test 결과와 분리하여 후속 실험 아이디어 및 한계로 기록한다.

## 10. 교수님 형식의 성능 Gap 리포팅

Gap은 모두 **양수일 때 뒤 단계의 성능이 더 나빠진 것**을 의미하도록 정의한다.

\[
	ext{Gap}_{Train-Valid}=	ext{MAPE}_{Valid}-\overline{	ext{MAPE}}_{Train\ CV}
\]

\[
	ext{Gap}_{Valid-Test}=	ext{MAPE}_{Test}-	ext{MAPE}_{Valid}
\]

\[
	ext{Gap}_{Target-Test}=	ext{MAPE}_{Test}-9.1
\]

MAPE 자체의 단위는 `%`, 두 MAPE의 차이인 Gap의 단위는 **퍼센트포인트(%p)**로 표기한다.

In [31]:
GAP_REPORT_VERSION = "v1"
GAP_REPORT_PATH = (
    MODEL_RESULT_DIR
    / f"professor_performance_report_{GAP_REPORT_VERSION}.csv"
)
GAP_DEFINITION_PATH = (
    MODEL_RESULT_DIR
    / f"gap_metrics_{GAP_REPORT_VERSION}.json"
)

train_cv_mape_mean = float(selected_cv_mape)
train_cv_mape_std = float(selected_cv_std)
valid_mape = float(selected_valid_metrics["mape_pct"])
test_mape_for_report = float(test_mape)

gap_train_valid = valid_mape - train_cv_mape_mean
gap_valid_test = test_mape_for_report - valid_mape
gap_target_test = test_mape_for_report - PAPER_TARGET_MAPE_PCT

# Section 8·9에서 저장한 값과 같은지 검사한다.
assert np.isclose(gap_train_valid, gap_cv_valid_pp)
assert np.isclose(gap_valid_test, gap_valid_test_pp)
assert np.isclose(gap_target_test, gap_target_test_pp)

professor_performance_report = pd.DataFrame([
    {
        "구분": "Train (Batch 1 CV)",
        "MAPE (%)": (
            f"{train_cv_mape_mean:.2f} ± "
            f"{train_cv_mape_std:.2f}"
        ),
        "비고": "Group CV",
    },
    {
        "구분": "Valid (Batch 1 Hold-out)",
        "MAPE (%)": f"{valid_mape:.2f}",
        "비고": "고정 policy Hold-out",
    },
    {
        "구분": "Test (Batch 2)",
        "MAPE (%)": f"{test_mape_for_report:.2f}",
        "비고": "최종 평가 1회",
    },
    {
        "구분": "Gap (Train–Valid)",
        "MAPE (%)": f"{gap_train_valid:+.2f}%p",
        "비고": "Valid − Train; 양수: 과적합 의심",
    },
    {
        "구분": "Gap (Valid–Test)",
        "MAPE (%)": f"{gap_valid_test:+.2f}%p",
        "비고": "Test − Valid; 양수: 배치 일반화 저하",
    },
    {
        "구분": "Gap (Target–Test)",
        "MAPE (%)": f"{gap_target_test:+.2f}%p",
        "비고": "Test − 9.1; 양수: 원논문보다 낮은 성능",
    },
])

gap_metrics_record = {
    "version": GAP_REPORT_VERSION,
    "primary_metric": "MAPE",
    "mape_unit": "%",
    "gap_unit": "%p",
    "positive_gap_meaning": "performance degradation at the later stage",
    "definitions": {
        "gap_train_valid": "valid_mape - train_cv_mape_mean",
        "gap_valid_test": "test_mape - valid_mape",
        "gap_target_test": "test_mape - 9.1",
    },
    "values": {
        "train_cv_mape_mean_pct": round(train_cv_mape_mean, 10),
        "train_cv_mape_std_pct": round(train_cv_mape_std, 10),
        "valid_mape_pct": round(valid_mape, 10),
        "test_mape_pct": round(test_mape_for_report, 10),
        "gap_train_valid_pp": round(gap_train_valid, 10),
        "gap_valid_test_pp": round(gap_valid_test, 10),
        "gap_target_test_pp": round(gap_target_test, 10),
    },
    "regression_metrics": ["MAPE", "MAE", "RMSE", "R2"],
    "classification_metrics_not_used": ["F1-score", "Accuracy"],
}

professor_performance_report.to_csv(GAP_REPORT_PATH, index=False)

if GAP_DEFINITION_PATH.exists():
    with GAP_DEFINITION_PATH.open("r", encoding="utf-8") as file:
        saved_gap_metrics_record = json.load(file)
    assert saved_gap_metrics_record == gap_metrics_record, (
        "저장된 Gap 정의·값과 현재 계산 결과가 다릅니다."
    )
    print("기존 Gap 계산 기록을 재사용합니다.")
else:
    with GAP_DEFINITION_PATH.open("w", encoding="utf-8") as file:
        json.dump(
            gap_metrics_record,
            file,
            ensure_ascii=False,
            indent=2,
        )
    print("Gap 계산 기록을 최초 생성했습니다.")

display(professor_performance_report)

print(f"Gap (Train–Valid): {gap_train_valid:+.2f}%p")
print(f"Gap (Valid–Test): {gap_valid_test:+.2f}%p")
print(f"Gap (Target–Test): {gap_target_test:+.2f}%p")
print(f"제출용 표 저장: {GAP_REPORT_PATH}")
print(f"Gap 정의 저장: {GAP_DEFINITION_PATH}")

Gap 계산 기록을 최초 생성했습니다.


,구분,MAPE (%),비고
0,Train (Batch 1 CV),7.60 ± 1.57,Group CV
1,Valid (Batch 1 Hold-out),11.63,고정 policy Hold-out
2,Test (Batch 2),31.53,최종 평가 1회
3,Gap (Train–Valid),+4.03%p,Valid − Train; 양수: 과적합 의심
4,Gap (Valid–Test),+19.90%p,Test − Valid; 양수: 배치 일반화 저하
5,Gap (Target–Test),+22.43%p,Test − 9.1; 양수: 원논문보다 낮은 성능


Gap (Train–Valid): +4.03%p
Gap (Valid–Test): +19.90%p
Gap (Target–Test): +22.43%p
제출용 표 저장: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/professor_performance_report_v1.csv
Gap 정의 저장: /Users/jungseungwoo/Desktop/data-mini-project/results/modeling/gap_metrics_v1.json


### 10.1 평가 지표 구분

스크린샷의 `Regression: F1-Score, Accuracy`는 오타로 판단한다.

- **Regression:** MAPE, MAE, RMSE, R²
- **Classification:** F1-score, Accuracy

이번 프로젝트의 목표변수 `cycle_life`는 연속형이므로 회귀 문제이며, **MAPE를 주 지표**로 사용한다. MAE·RMSE·R²는 오차 규모와 설명력을 보완해서 해석하는 보조 지표로 함께 보고한다.